# Portfolio inspection copy: CW1 Fraud

Adapted from the completed coursework notebook. Embedded historical outputs and cloud metadata were removed for privacy and data redistribution control. The original notebook and its outputs are preserved unchanged outside this repository. Verified aggregate results are under `Project1/results`, `Project2/final_evaluation` or `CW2/results`.

Every original code cell is behind `PORTFOLIO_EXECUTE=False`, so Run All in the default state neither imports the ML stack nor trains, evaluates, installs packages, mounts Drive or writes files. Read code and saved result files first. See `../docs/REPRODUCIBILITY.md` before opting in. CW2 combines fitting and test scoring in one historical workflow; it requires all three opt-ins. No execution was used to regenerate the recorded metrics.


In [ ]:
PORTFOLIO_EXECUTE = False
RUN_TRAINING = False
RUN_FINAL_TEST = False
from pathlib import Path
REPO_ROOT = Path.cwd() if (Path.cwd()/"Project1").is_dir() else Path.cwd().parent


# **PROJECT 1: FINTECH SECURITY NEURAL FRAUD DETECTION**

This submission copy preserves the original executed outputs and locked E0-E5 evidence. `SUBMISSION_MODE=True` prevents file writes, model fitting, prediction, and final evaluation during inspection. Set it to `False` only for an intentional full reproduction using the documented dataset and Google Drive paths.


**1. Environment and Reproducibility**

In [ ]:
if PORTFOLIO_EXECUTE:
    print("Using repository-local files; no Drive mount is required.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # Section 1: Environment and Reproducibility
    # ============================================================

    import sys
    import random
    import numpy as np
    import pandas as pd
    import sklearn
    import tensorflow as tf
    import matplotlib.pyplot as plt

    # Fixed seed for reproducibility
    SEED = 42

    random.seed(SEED)
    np.random.seed(SEED)
    tf.random.set_seed(SEED)

    # Enable deterministic TensorFlow operations where supported
    try:
        tf.config.experimental.enable_op_determinism()
        deterministic_status = "Enabled"
    except Exception as e:
        deterministic_status = f"Not enabled: {e}"

    print("Python version:", sys.version.split()[0])
    print("Pandas version:", pd.__version__)
    print("NumPy version:", np.__version__)
    print("Scikit-learn version:", sklearn.__version__)
    print("TensorFlow version:", tf.__version__)
    print("Random seed:", SEED)
    print("TensorFlow deterministic operations:", deterministic_status)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    from pathlib import Path

    # ============================================================
    # M32366 OIDS Coursework 1 - Permanent Project Directories
    # ============================================================

    CW1_DIR = REPO_ROOT

    DATA_PATH = CW1_DIR / "Project1/data/creditcard.csv"

    PROJECT1_DIR = CW1_DIR / "Project1"

    MODEL_DIR = PROJECT1_DIR / "models"
    FIGURE_DIR = PROJECT1_DIR / "figures"
    RESULTS_DIR = PROJECT1_DIR / "results"
    LOG_DIR = PROJECT1_DIR / "experiment_logs"

    # Create permanent folders
    for folder in [
        PROJECT1_DIR,
        MODEL_DIR,
        FIGURE_DIR,
        RESULTS_DIR,
        LOG_DIR
    ]:
        folder.mkdir(parents=True, exist_ok=True)

    print("CW1 directory:", CW1_DIR)
    print("Dataset:", DATA_PATH)
    print("Dataset exists:", DATA_PATH.exists())

    print("\nPermanent Project 1 folders:")
    print("Models:", MODEL_DIR)
    print("Figures:", FIGURE_DIR)
    print("Results:", RESULTS_DIR)
    print("Experiment logs:", LOG_DIR)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # Runtime / Resume Configuration
    # ============================================================

    from types import SimpleNamespace

    FORCE_RETRAIN = False

    print("Force retraining:", FORCE_RETRAIN)
else:
    print("Inspection mode: code not executed.")


**2. Dataset Acquisition**

**2.1 Dataset Source and Structure**

This project uses the Kaggle **Credit Card Fraud Detection** dataset produced by the Machine Learning Group at Université Libre de Bruxelles (ULB). The dataset contains European card transactions, with `Class` as the binary target variable (`0 = legitimate`, `1 = fraud`).

The predictor variables comprise `Time`, `Amount`, and 28 anonymised PCA-transformed features (`V1`–`V28`). The PCA transformation protects the confidentiality of the original transaction attributes while limiting direct interpretation of the transformed features.

In [ ]:
if PORTFOLIO_EXECUTE:
    import pandas as pd

    df = pd.read_csv(DATA_PATH)

    print("Dataset loaded successfully.")
    print(f"Rows: {df.shape[0]:,}")
    print(f"Columns: {df.shape[1]}")
    print(f"Dataset shape: {df.shape}")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    df.head()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    print(df.columns.tolist())
else:
    print("Inspection mode: code not executed.")


**3. Data Integrity Assessment**

Before preprocessing or model development, the dataset was examined for structural and data-quality issues including missing values, infinite values, duplicated records, data types and class imbalance. The dataset was first assessed without modification. Exact duplicate records were then examined separately and removed before data splitting to reduce the risk of identical observations appearing across training and evaluation subsets.

In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 3.1 Dataset Structure
    # ============================================================

    print(f"Number of transactions: {df.shape[0]:,}")
    print(f"Number of features/columns: {df.shape[1]}")

    print("\nColumn data types:")
    print(df.dtypes.value_counts())

    print("\nTarget values:")
    print(sorted(df["Class"].unique()))
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 3.2 Missing Values
    # ============================================================

    missing_by_column = df.isnull().sum()
    total_missing = int(missing_by_column.sum())

    print(f"Total missing values: {total_missing:,}")
    print(f"Columns containing missing values: {(missing_by_column > 0).sum()}")

    if total_missing > 0:
        display(missing_by_column[missing_by_column > 0].to_frame("Missing Values"))
    else:
        print("No missing values detected.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 3.3 Infinite Values
    # ============================================================

    numeric_df = df.select_dtypes(include=[np.number])

    infinite_by_column = np.isinf(numeric_df).sum()
    total_infinite = int(infinite_by_column.sum())

    print(f"Total infinite values: {total_infinite:,}")
    print(f"Columns containing infinite values: {(infinite_by_column > 0).sum()}")

    if total_infinite > 0:
        display(infinite_by_column[infinite_by_column > 0].to_frame("Infinite Values"))
    else:
        print("No infinite values detected.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 3.4 Duplicate Records
    # ============================================================

    duplicate_count = int(df.duplicated().sum())
    duplicate_percentage = (duplicate_count / len(df)) * 100

    print(f"Exact duplicate rows: {duplicate_count:,}")
    print(f"Duplicate percentage: {duplicate_percentage:.4f}%")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # Inspect duplicate records before deciding whether to remove them

    duplicate_rows = df[df.duplicated(keep=False)]

    print(f"Total rows involved in duplicate groups: {len(duplicate_rows):,}")

    print("\nClass distribution among duplicated rows:")
    print(duplicate_rows["Class"].value_counts().sort_index())

    print("\nPercentage distribution among duplicated rows:")
    print(
        duplicate_rows["Class"]
        .value_counts(normalize=True)
        .sort_index() * 100
    )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # Compare overall class distribution before any duplicate removal

    class_counts = df["Class"].value_counts().sort_index()
    class_percentages = df["Class"].value_counts(
        normalize=True
    ).sort_index() * 100

    class_summary = pd.DataFrame({
        "Count": class_counts,
        "Percentage (%)": class_percentages
    })

    class_summary.index = ["Legitimate (0)", "Fraud (1)"]

    display(class_summary)

    legitimate_count = int(class_counts.loc[0])
    fraud_count = int(class_counts.loc[1])
    fraud_rate = float(class_percentages.loc[1])
    class_ratio = legitimate_count / fraud_count

    print(f"Legitimate transactions: {legitimate_count:,}")
    print(f"Fraudulent transactions: {fraud_count:,}")
    print(f"Fraud rate: {fraud_rate:.4f}%")
    print(f"Approximate ratio: 1 fraud : {class_ratio:.0f} legitimate")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 3.4.1 Impact of Removing Exact Duplicate Records
    # ============================================================

    # Rows that would actually be removed when keeping the first occurrence
    rows_to_remove = df[df.duplicated(keep="first")]

    print("Rows that would be removed:")
    print(rows_to_remove["Class"].value_counts().sort_index())

    print(f"\nTotal rows removed: {len(rows_to_remove):,}")

    # Create cleaned working dataset while preserving original df
    df_clean = df.drop_duplicates(keep="first").reset_index(drop=True)

    print(f"\nOriginal dataset size: {len(df):,}")
    print(f"Cleaned dataset size: {len(df_clean):,}")

    # Compare class distributions before and after duplicate removal
    before_counts = df["Class"].value_counts().sort_index()
    after_counts = df_clean["Class"].value_counts().sort_index()

    dedup_comparison = pd.DataFrame({
        "Before Removal": before_counts,
        "After Removal": after_counts,
        "Removed": before_counts - after_counts
    })

    dedup_comparison.index = ["Legitimate (0)", "Fraud (1)"]

    display(dedup_comparison)

    print(
        "\nFraud percentage before removal:",
        f"{df['Class'].mean() * 100:.4f}%"
    )

    print(
        "Fraud percentage after removal:",
        f"{df_clean['Class'].mean() * 100:.4f}%"
    )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 19; saved output below remains the submitted evidence.')
    else:
        # Save duplicate-removal comparison to Google Drive
    
        dedup_path = RESULTS_DIR / "P1_duplicate_removal_impact.csv"
        dedup_comparison.to_csv(dedup_path)
    
        print("Duplicate-removal analysis saved to:")
        print(dedup_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 20; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save Final Data Integrity Summary
        # ============================================================
    
        clean_legitimate = int(
            (df_clean["Class"] == 0).sum()
        )
    
        clean_fraud = int(
            (df_clean["Class"] == 1).sum()
        )
    
        clean_fraud_rate = (
            clean_fraud / len(df_clean)
        ) * 100
    
        clean_ratio = (
            clean_legitimate / clean_fraud
        )
    
        # Calculate removed duplicate counts directly from the data
        removed_class_counts = (
            rows_to_remove["Class"]
            .value_counts()
            .reindex(
                [0, 1],
                fill_value=0
            )
        )
    
        removed_legitimate = int(
            removed_class_counts.loc[0]
        )
    
        removed_fraud = int(
            removed_class_counts.loc[1]
        )
    
        integrity_summary = pd.DataFrame({
            "Measure": [
                "Original transactions",
                "Missing values",
                "Infinite values",
                "Exact duplicates removed",
                "Legitimate duplicates removed",
                "Fraud duplicates removed",
                "Cleaned transactions",
                "Cleaned legitimate transactions",
                "Cleaned fraudulent transactions",
                "Cleaned fraud percentage",
                "Legitimate-to-fraud ratio"
            ],
            "Value": [
                len(df),
                total_missing,
                total_infinite,
                len(rows_to_remove),
                removed_legitimate,
                removed_fraud,
                len(df_clean),
                clean_legitimate,
                clean_fraud,
                round(clean_fraud_rate, 4),
                round(clean_ratio, 2)
            ]
        })
    
        display(integrity_summary)
    
        integrity_summary.to_csv(
            RESULTS_DIR /
            "P1_data_integrity_summary.csv",
            index=False
        )
    
        print(
            "Data integrity summary saved successfully."
        )
else:
    print("Inspection mode: code not executed.")


**4. Exploratory Data Analysis**

Exploratory analysis was conducted on the cleaned dataset to examine the severity of class imbalance and the distributions of the available transaction variables before preprocessing and model development.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 22; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 4.1 Cleaned Class Distribution - Final Report Figure
        # ============================================================
    
        class_counts_clean = df_clean["Class"].value_counts().sort_index()
    
        legitimate_clean = int(class_counts_clean.loc[0])
        fraud_clean = int(class_counts_clean.loc[1])
    
        total_clean = len(df_clean)
    
        legitimate_pct = (legitimate_clean / total_clean) * 100
        fraud_pct = (fraud_clean / total_clean) * 100
    
        labels = ["Legitimate", "Fraud"]
        values = [legitimate_clean, fraud_clean]
        percentages = [legitimate_pct, fraud_pct]
    
        fig, ax = plt.subplots(figsize=(8, 5.5))
    
        bars = ax.bar(labels, values)
    
        # Logarithmic scale to make the minority fraud class visible
        ax.set_yscale("log")
    
        # Extra headroom prevents labels being clipped
        ax.set_ylim(300, 800000)
    
        ax.set_title(
            "Class Distribution of Cleaned Credit Card Transactions",
            pad=18
        )
    
        ax.set_ylabel("Number of Transactions (Log Scale)")
        ax.set_xlabel("Transaction Class")
    
        for bar, value, pct in zip(bars, values, percentages):
            ax.annotate(
                f"{value:,}\n({pct:.4f}%)",
                xy=(bar.get_x() + bar.get_width() / 2, value),
                xytext=(0, 6),
                textcoords="offset points",
                ha="center",
                va="bottom"
            )
    
        plt.tight_layout()
    
        figure_path = FIGURE_DIR / "P1_01_class_distribution.png"
    
        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Figure saved to:")
        print(figure_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 4.2 Transaction Amount Distribution
    # ============================================================

    amount_summary = df_clean["Amount"].describe()

    display(amount_summary.to_frame("Amount"))

    print(f"Median transaction amount: {df_clean['Amount'].median():.2f}")
    print(f"Maximum transaction amount: {df_clean['Amount'].max():.2f}")
    print(f"Skewness of transaction amount: {df_clean['Amount'].skew():.4f}")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 24; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Figure P1-02: Transaction Amount Distribution
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.hist(
            df_clean["Amount"],
            bins=100
        )
    
        ax.set_title("Distribution of Credit Card Transaction Amounts")
        ax.set_xlabel("Transaction Amount")
        ax.set_ylabel("Number of Transactions")
    
        plt.tight_layout()
    
        amount_path = FIGURE_DIR / "P1_02_amount_distribution.png"
    
        plt.savefig(
            amount_path,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Figure saved to:")
        print(amount_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 4.3 Transaction Amount by Class
    # ============================================================

    amount_by_class = (
        df_clean
        .groupby("Class")["Amount"]
        .agg(["count", "mean", "median", "std", "min", "max"])
    )

    amount_by_class.index = ["Legitimate", "Fraud"]

    display(amount_by_class)

    print(
        f"Median legitimate transaction amount: "
        f"{df_clean.loc[df_clean['Class'] == 0, 'Amount'].median():.2f}"
    )

    print(
        f"Median fraudulent transaction amount: "
        f"{df_clean.loc[df_clean['Class'] == 1, 'Amount'].median():.2f}"
    )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 26; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 4.4 Transaction Amount by Class - Log-Transformed View
        # ============================================================
    
        legitimate_amount_log = np.log1p(
            df_clean.loc[df_clean["Class"] == 0, "Amount"]
        )
    
        fraud_amount_log = np.log1p(
            df_clean.loc[df_clean["Class"] == 1, "Amount"]
        )
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.boxplot(
            [legitimate_amount_log, fraud_amount_log],
           tick_labels=["Legitimate", "Fraud"],
            showfliers=False
        )
    
        ax.set_title("Transaction Amount Distribution by Class")
        ax.set_xlabel("Transaction Class")
        ax.set_ylabel("log(1 + Transaction Amount)")
    
        plt.tight_layout()
    
        amount_class_path = FIGURE_DIR / "P1_02_amount_by_class.png"
    
        plt.savefig(
            amount_class_path,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Figure saved to:")
        print(amount_class_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 27; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save Transaction Amount EDA Results
        # ============================================================
    
        amount_summary.to_frame("Amount").to_csv(
            RESULTS_DIR / "P1_amount_summary.csv"
        )
    
        amount_by_class.to_csv(
            RESULTS_DIR / "P1_amount_by_class.csv"
        )
    
        print("Amount EDA results saved successfully.")
else:
    print("Inspection mode: code not executed.")


**5. 70/30 Stratified Data Split**

The cleaned dataset was divided into training and test subsets using a stratified 70/30 split in order to preserve the highly imbalanced class distribution across both subsets. A validation subset was then created from the training data for model tuning and threshold selection, while the held-out test set was not used for model or threshold selection and was reserved for final evaluation.

In [ ]:
if PORTFOLIO_EXECUTE:
    from sklearn.model_selection import train_test_split
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 5.1 Features and Target
    # ============================================================

    X = df_clean.drop("Class", axis=1)
    y = df_clean["Class"]

    print("Feature matrix shape:", X.shape)
    print("Target vector shape:", y.shape)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 5.2 70/30 Stratified Train-Test Split
    # ============================================================

    X_train_full, X_test, y_train_full, y_test = train_test_split(
        X,
        y,
        test_size=0.30,
        stratify=y,
        random_state=SEED
    )

    print("Training set shape:", X_train_full.shape)
    print("Test set shape:", X_test.shape)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 5.3 Train-Validation Split from Training Data
    # ============================================================

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full,
        y_train_full,
        test_size=0.20,
        stratify=y_train_full,
        random_state=SEED
    )

    print("Final training set shape:", X_train.shape)
    print("Validation set shape:", X_val.shape)
    print("Final test set shape:", X_test.shape)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 5.4 Verify Split Independence
    # ============================================================

    train_indices = set(X_train.index)
    val_indices = set(X_val.index)
    test_indices = set(X_test.index)

    train_val_overlap = len(
        train_indices.intersection(val_indices)
    )

    train_test_overlap = len(
        train_indices.intersection(test_indices)
    )

    val_test_overlap = len(
        val_indices.intersection(test_indices)
    )

    print("Train-validation overlap:", train_val_overlap)
    print("Train-test overlap:", train_test_overlap)
    print("Validation-test overlap:", val_test_overlap)

    if (
        train_val_overlap == 0
        and train_test_overlap == 0
        and val_test_overlap == 0
    ):
        print("PASS: Training, validation and test subsets are fully disjoint.")
    else:
        print("WARNING: Overlapping observations detected.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    #  5.5 Class Distribution Across Splits
    # ============================================================

    def summarise_split(y_split, split_name):
        counts = y_split.value_counts().sort_index()
        percentages = y_split.value_counts(normalize=True).sort_index() * 100

        summary = pd.DataFrame({
            "Count": counts,
            "Percentage (%)": percentages
        })

        summary.index = ["Legitimate (0)", "Fraud (1)"]

        print(f"\n{split_name}")
        display(summary)

        fraud_rate = percentages.loc[1]
        print(f"{split_name} fraud rate: {fraud_rate:.4f}%")

    summarise_split(y_train, "Training Set")
    summarise_split(y_val, "Validation Set")
    summarise_split(y_test, "Test Set")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 35; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save Split Summary
        # ============================================================
    
        split_summary = pd.DataFrame({
            "Subset": ["Train", "Validation", "Test"],
            "Rows": [len(X_train), len(X_val), len(X_test)],
            "Legitimate": [
                int((y_train == 0).sum()),
                int((y_val == 0).sum()),
                int((y_test == 0).sum())
            ],
            "Fraud": [
                int((y_train == 1).sum()),
                int((y_val == 1).sum()),
                int((y_test == 1).sum())
            ]
        })
    
        split_summary["Fraud Percentage"] = (
            split_summary["Fraud"] / split_summary["Rows"] * 100
        )
    
        display(split_summary)
    
        split_summary.to_csv(
            RESULTS_DIR / "P1_split_summary.csv",
            index=False
        )
    
        print("Split summary saved successfully.")
else:
    print("Inspection mode: code not executed.")


**6. Preprocessing**

Preprocessing parameters were learned exclusively from the training subset to prevent information from the validation or test datasets influencing model development. StandardScaler was selected as the baseline scaling method. RobustScaler will subsequently be evaluated as a separate experiment due to the substantial skewness and extreme values identified in the transaction amount variable.

In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 6.1 Baseline Standardisation
    # ============================================================

    from sklearn.preprocessing import StandardScaler
    import joblib

    standard_scaler = StandardScaler()

    # Fit ONLY on the training data
    X_train_scaled = standard_scaler.fit_transform(X_train)

    # Apply the training-derived transformation
    X_val_scaled = standard_scaler.transform(X_val)
    X_test_scaled = standard_scaler.transform(X_test)

    print("Standardisation completed.")

    print("\nScaled dataset shapes:")
    print("Training:", X_train_scaled.shape)
    print("Validation:", X_val_scaled.shape)
    print("Test:", X_test_scaled.shape)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # 6.2 Verify Standardisation
    # ============================================================

    print(
        "Mean of scaled training features:",
        round(float(X_train_scaled.mean()), 6)
    )

    print(
        "Standard deviation of scaled training features:",
        round(float(X_train_scaled.std()), 6)
    )

    print("\nNaN values after scaling:",
          np.isnan(X_train_scaled).sum())

    print(
        "Infinite values after scaling:",
        np.isinf(X_train_scaled).sum()
    )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 39; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save Baseline StandardScaler
        # ============================================================
    
        scaler_path = MODEL_DIR / "P1_standard_scaler.joblib"
    
        joblib.dump(
            standard_scaler,
            scaler_path
        )
    
        print("StandardScaler saved to:")
        print(scaler_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    # ============================================================
    # Save Exact Dataset Split Indices
    # ============================================================

    split_indices = {
        "train_indices": X_train.index.to_numpy(),
        "validation_indices": X_val.index.to_numpy(),
        "test_indices": X_test.index.to_numpy()
    }

    np.savez(
        RESULTS_DIR / "P1_split_indices.npz",
        **split_indices
    )

    print("Exact split indices saved successfully.")
else:
    print("Inspection mode: code not executed.")


**7. Experiment E0 – Baseline DNN**

Experiment E0 establishes a reference model using the standardised training data without any class-imbalance treatment. The model uses two fully connected hidden layers with ReLU activation and a sigmoid output for binary classification. Its purpose is to demonstrate baseline behaviour before cost-sensitive learning or oversampling is introduced.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 42; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.1 Experiment E0 - Required Evaluation Tools
        # ============================================================
    
        from sklearn.metrics import (
            accuracy_score,
            precision_score,
            recall_score,
            f1_score,
            confusion_matrix,
            ConfusionMatrixDisplay,
            precision_recall_curve,
            average_precision_score,
            roc_auc_score,
            auc
        )
    
        from tensorflow.keras import Sequential
    
        from tensorflow.keras.layers import (
            Input,
            Dense
        )
    
        from tensorflow.keras.callbacks import (
            EarlyStopping
        )
    
        print("Evaluation and baseline DNN tools imported successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 43; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.2 Build Baseline DNN
        # ============================================================
    
        # Clear any previous TensorFlow state
        tf.keras.backend.clear_session()
    
        # Reset seeds for reproducibility
        random.seed(SEED)
        np.random.seed(SEED)
        tf.random.set_seed(SEED)
    
        baseline_model = Sequential([
            Input(shape=(X_train_scaled.shape[1],)),
    
            Dense(
                64,
                activation="relu",
                name="hidden_layer_1"
            ),
    
            Dense(
                32,
                activation="relu",
                name="hidden_layer_2"
            ),
    
            Dense(
                1,
                activation="sigmoid",
                name="fraud_probability"
            )
        ])
    
        baseline_model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=0.001
            ),
            loss="binary_crossentropy",
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.Precision(name="precision"),
                tf.keras.metrics.Recall(name="recall"),
                tf.keras.metrics.AUC(
                    curve="PR",
                    name="pr_auc"
                )
            ]
        )
    
        baseline_model.summary()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 44; saved output below remains the submitted evidence.')
    else:
        # Save architecture summary to Google Drive
    
        summary_path = RESULTS_DIR / "P1_E0_model_summary.txt"
    
        with open(summary_path, "w") as f:
            baseline_model.summary(
                print_fn=lambda line: f.write(line + "\n")
            )
    
        print("E0 model summary saved to:")
        print(summary_path)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 45; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.3 Early Stopping
        # ============================================================
    
        early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 46; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.4 Train OR Restore Experiment E0
        # ============================================================
    
        E0_EPOCHS = 30
        E0_BATCH_SIZE = 1024
    
        E0_MODEL_PATH = MODEL_DIR / "P1_E0_baseline_dnn.keras"
        E0_HISTORY_PATH = RESULTS_DIR / "P1_E0_training_history.csv"
    
        if (
            E0_MODEL_PATH.exists()
            and E0_HISTORY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E0 experiment found.")
            print("Loading saved E0 model and history instead of retraining...")
    
            baseline_model = tf.keras.models.load_model(E0_MODEL_PATH)
    
            history_E0_df = pd.read_csv(
                E0_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E0 = SimpleNamespace(
                history=history_E0_df.to_dict(orient="list")
            )
    
            print("E0 restored successfully.")
    
        else:
            print("Training E0 from scratch...")
    
            history_E0 = baseline_model.fit(
                X_train_scaled,
                y_train,
                validation_data=(X_val_scaled, y_val),
                epochs=E0_EPOCHS,
                batch_size=E0_BATCH_SIZE,
                callbacks=[early_stopping],
                verbose=1
            )
    
            baseline_model.save(E0_MODEL_PATH)
    
            history_E0_df = pd.DataFrame(history_E0.history)
            history_E0_df.index.name = "Epoch"
    
            history_E0_df.to_csv(E0_HISTORY_PATH)
    
            print("E0 training completed and saved.")
    
        print("E0 model path:", E0_MODEL_PATH)
        print("E0 history path:", E0_HISTORY_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 47; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.5 E0 Validation Predictions
        # ============================================================
    
        E0_val_probabilities = baseline_model.predict(
            X_val_scaled,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E0_THRESHOLD = 0.50
    
        E0_val_predictions = (
            E0_val_probabilities >= E0_THRESHOLD
        ).astype(int)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 48; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.6 E0 Validation Metrics
        # ============================================================
    
        E0_accuracy = accuracy_score(
            y_val,
            E0_val_predictions
        )
    
        E0_precision = precision_score(
            y_val,
            E0_val_predictions,
            zero_division=0
        )
    
        E0_recall = recall_score(
            y_val,
            E0_val_predictions,
            zero_division=0
        )
    
        E0_f1 = f1_score(
            y_val,
            E0_val_predictions,
            zero_division=0
        )
    
        precision_curve_E0, recall_curve_E0, thresholds_E0 = (
            precision_recall_curve(
                y_val,
                E0_val_probabilities
            )
        )
    
        E0_pr_auc = auc(
            recall_curve_E0,
            precision_curve_E0
        )
    
        E0_average_precision = average_precision_score(
            y_val,
            E0_val_probabilities
        )
    
        E0_roc_auc = roc_auc_score(
            y_val,
            E0_val_probabilities
        )
    
        E0_tn, E0_fp, E0_fn, E0_tp = confusion_matrix(
            y_val,
            E0_val_predictions
        ).ravel()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 49; saved output below remains the submitted evidence.')
    else:
        E0_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E0_accuracy,
                E0_precision,
                E0_recall,
                E0_f1,
                E0_pr_auc,
                E0_average_precision,
                E0_roc_auc,
                E0_tn,
                E0_fp,
                E0_fn,
                E0_tp
            ]
        })
    
        display(E0_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 50; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E0 Validation Predictions
        # ============================================================
    
        E0_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E0_val_probabilities,
            "Predicted_Class_0.50": E0_val_predictions
        })
    
        E0_predictions_df.to_csv(
            RESULTS_DIR / "P1_E0_validation_predictions.csv",
            index=False
        )
    
        print("E0 validation predictions saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 51; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save / Update Master Experiment Results
        # ============================================================
    
        E0_result_row = pd.DataFrame([{
            "Experiment": "E0",
            "Description": "Baseline DNN",
            "Scaler": "StandardScaler",
            "Imbalance_Method": "None",
            "Architecture": "64-32-1",
            "Threshold": 0.50,
            "Accuracy": E0_accuracy,
            "Precision": E0_precision,
            "Recall": E0_recall,
            "F1": E0_f1,
            "PR_AUC": E0_pr_auc,
            "Average_Precision": E0_average_precision,
            "ROC_AUC": E0_roc_auc,
            "TN": E0_tn,
            "FP": E0_fp,
            "FN": E0_fn,
            "TP": E0_tp
        }])
    
        experiment_results_path = (
            RESULTS_DIR / "P1_experiment_results.csv"
        )
    
        if experiment_results_path.exists():
            experiment_results = pd.read_csv(
                experiment_results_path
            )
    
            experiment_results = pd.concat(
                [experiment_results, E0_result_row],
                ignore_index=True
            )
    
            # Prevent accidental duplicate E0 records when rerunning
            experiment_results = (
                experiment_results
                .drop_duplicates(
                    subset="Experiment",
                    keep="last"
                )
            )
    
        else:
            experiment_results = E0_result_row.copy()
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("Master experiment results saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 52; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Initialise / Update Project 1 Experiment Log
        # ============================================================
    
        E0_log_row = pd.DataFrame([{
            "Experiment": "E0",
            "Change": "Baseline model; no imbalance correction",
            "Hypothesis":
                "A conventional DNN will provide a reference point but "
                "may favour the overwhelming legitimate class.",
            "Observation":
                "To be interpreted from validation metrics.",
            "Decision":
                "Retain as baseline for comparison with class weighting "
                "and SMOTE."
        }])
    
        experiment_log_path = (
            LOG_DIR / "P1_experiment_log.csv"
        )
    
        if experiment_log_path.exists():
            experiment_log = pd.read_csv(
                experiment_log_path
            )
    
            experiment_log = pd.concat(
                [experiment_log, E0_log_row],
                ignore_index=True
            )
    
            experiment_log = (
                experiment_log
                .drop_duplicates(
                    subset="Experiment",
                    keep="last"
                )
            )
    
        else:
            experiment_log = E0_log_row.copy()
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E0 experiment log saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 53; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E0 Experiment Log with Actual Findings
        # ============================================================
    
        experiment_log.loc[
            experiment_log["Experiment"] == "E0",
            "Observation"
        ] = (
            "At the 0.50 threshold, the baseline DNN achieved 99.9345% "
            "validation accuracy, but recall was 69.70%. It detected "
            "46 of 66 fraudulent transactions while missing 20 "
            "(30.30% of fraud cases). Precision was 88.46%, F1 was "
            "0.7797 and PR-AUC was 0.8166. The very high accuracy "
            "therefore overstated the model's effectiveness for fraud detection."
        )
    
        experiment_log.loc[
            experiment_log["Experiment"] == "E0",
            "Decision"
        ] = (
            "Retain E0 as the untreated baseline. Proceed to cost-sensitive "
            "learning to investigate whether fraud recall can be improved "
            "without an unacceptable increase in false-positive alerts."
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E0 experiment log updated successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 54; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        #  7.7 E0 Training and Validation Loss
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.plot(
            history_E0_df.index + 1,
            history_E0_df["loss"],
            label="Training Loss"
        )
    
        ax.plot(
            history_E0_df.index + 1,
            history_E0_df["val_loss"],
            label="Validation Loss"
        )
    
        ax.set_title("E0 Baseline DNN – Training and Validation Loss")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Binary Cross-Entropy Loss")
        ax.legend()
    
        plt.tight_layout()
    
        plt.savefig(
            FIGURE_DIR / "P1_E0_loss.png",
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 55; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 7.8 E0 Training and Validation Accuracy
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.plot(
            history_E0_df.index + 1,
            history_E0_df["accuracy"],
            label="Training Accuracy"
        )
    
        ax.plot(
            history_E0_df.index + 1,
            history_E0_df["val_accuracy"],
            label="Validation Accuracy"
        )
    
        ax.set_title("E0 Baseline DNN – Training and Validation Accuracy")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy")
        ax.legend()
    
        plt.tight_layout()
    
        plt.savefig(
            FIGURE_DIR / "P1_E0_accuracy.png",
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
else:
    print("Inspection mode: code not executed.")


**8. Experiment E1 – Cost-Sensitive DNN**

Experiment E1 evaluates cost-sensitive learning as an alternative approach to managing extreme class imbalance. The preprocessing pipeline, DNN architecture, optimiser and training configuration are retained from E0. The only major change is the introduction of class weights so that errors involving the minority fraud class contribute more strongly to the training objective.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 57; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 8.1 Calculate Balanced Class Weights
        # ============================================================
    
        from sklearn.utils.class_weight import compute_class_weight
    
        classes = np.array([0, 1])
    
        class_weights_array = compute_class_weight(
            class_weight="balanced",
            classes=classes,
            y=y_train
        )
    
        E1_CLASS_WEIGHTS = {
            0: float(class_weights_array[0]),
            1: float(class_weights_array[1])
        }
    
        print("E1 Class Weights:")
        print(f"Legitimate class (0): {E1_CLASS_WEIGHTS[0]:.4f}")
        print(f"Fraud class (1):      {E1_CLASS_WEIGHTS[1]:.4f}")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 58; saved output below remains the submitted evidence.')
    else:
        E1_class_weights_df = pd.DataFrame({
            "Class": ["Legitimate (0)", "Fraud (1)"],
            "Weight": [
                E1_CLASS_WEIGHTS[0],
                E1_CLASS_WEIGHTS[1]
            ]
        })
    
        display(E1_class_weights_df)
    
        E1_class_weights_df.to_csv(
            RESULTS_DIR / "P1_E1_class_weights.csv",
            index=False
        )
    
        print("E1 class weights saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 59; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 8.2 Build E1 Cost-Sensitive DNN
        # ============================================================
    
        tf.keras.backend.clear_session()
    
        random.seed(SEED)
        np.random.seed(SEED)
        tf.random.set_seed(SEED)
    
        E1_model = Sequential([
            Input(shape=(X_train_scaled.shape[1],)),
    
            Dense(
                64,
                activation="relu",
                name="hidden_layer_1"
            ),
    
            Dense(
                32,
                activation="relu",
                name="hidden_layer_2"
            ),
    
            Dense(
                1,
                activation="sigmoid",
                name="fraud_probability"
            )
        ])
    
        E1_model.compile(
            optimizer=tf.keras.optimizers.Adam(
                learning_rate=0.001
            ),
            loss="binary_crossentropy",
            metrics=[
                tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                tf.keras.metrics.Precision(name="precision"),
                tf.keras.metrics.Recall(name="recall"),
                tf.keras.metrics.AUC(
                    curve="PR",
                    name="pr_auc"
                )
            ]
        )
    
        E1_model.summary()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 60; saved output below remains the submitted evidence.')
    else:
        E1_early_stopping = EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
            verbose=1
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 61; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 8.3 Train OR Restore Experiment E1
        # ============================================================
    
        E1_EPOCHS = 30
        E1_BATCH_SIZE = 1024
    
        E1_MODEL_PATH = MODEL_DIR / "P1_E1_cost_sensitive_dnn.keras"
        E1_HISTORY_PATH = RESULTS_DIR / "P1_E1_training_history.csv"
    
        if (
            E1_MODEL_PATH.exists()
            and E1_HISTORY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E1 experiment found.")
            print("Loading saved E1 model and history instead of retraining...")
    
            E1_model = tf.keras.models.load_model(E1_MODEL_PATH)
    
            history_E1_df = pd.read_csv(
                E1_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E1 = SimpleNamespace(
                history=history_E1_df.to_dict(orient="list")
            )
    
            print("E1 restored successfully.")
    
        else:
            print("Training E1 from scratch...")
    
            history_E1 = E1_model.fit(
                X_train_scaled,
                y_train,
                validation_data=(X_val_scaled, y_val),
                epochs=E1_EPOCHS,
                batch_size=E1_BATCH_SIZE,
                class_weight=E1_CLASS_WEIGHTS,
                callbacks=[E1_early_stopping],
                verbose=1
            )
    
            E1_model.save(E1_MODEL_PATH)
    
            history_E1_df = pd.DataFrame(history_E1.history)
            history_E1_df.index.name = "Epoch"
    
            history_E1_df.to_csv(E1_HISTORY_PATH)
    
            print("E1 training completed and saved.")
    
        print("E1 model path:", E1_MODEL_PATH)
        print("E1 history path:", E1_HISTORY_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 62; saved output below remains the submitted evidence.')
    else:
        E1_val_probabilities = E1_model.predict(
            X_val_scaled,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E1_THRESHOLD = 0.50
    
        E1_val_predictions = (
            E1_val_probabilities >= E1_THRESHOLD
        ).astype(int)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 63; saved output below remains the submitted evidence.')
    else:
        E1_accuracy = accuracy_score(
            y_val,
            E1_val_predictions
        )
    
        E1_precision = precision_score(
            y_val,
            E1_val_predictions,
            zero_division=0
        )
    
        E1_recall = recall_score(
            y_val,
            E1_val_predictions,
            zero_division=0
        )
    
        E1_f1 = f1_score(
            y_val,
            E1_val_predictions,
            zero_division=0
        )
    
        precision_curve_E1, recall_curve_E1, thresholds_E1 = (
            precision_recall_curve(
                y_val,
                E1_val_probabilities
            )
        )
    
        E1_pr_auc = auc(
            recall_curve_E1,
            precision_curve_E1
        )
    
        E1_average_precision = average_precision_score(
            y_val,
            E1_val_probabilities
        )
    
        E1_roc_auc = roc_auc_score(
            y_val,
            E1_val_probabilities
        )
    
        E1_tn, E1_fp, E1_fn, E1_tp = confusion_matrix(
            y_val,
            E1_val_predictions
        ).ravel()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 64; saved output below remains the submitted evidence.')
    else:
        E1_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E1_accuracy,
                E1_precision,
                E1_recall,
                E1_f1,
                E1_pr_auc,
                E1_average_precision,
                E1_roc_auc,
                E1_tn,
                E1_fp,
                E1_fn,
                E1_tp
            ]
        })
    
        display(E1_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 65; saved output below remains the submitted evidence.')
    else:
        E0_E1_comparison = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1",
                "PR-AUC",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "E0 Baseline": [
                E0_accuracy,
                E0_precision,
                E0_recall,
                E0_f1,
                E0_pr_auc,
                E0_fp,
                E0_fn,
                E0_tp
            ],
            "E1 Class Weight": [
                E1_accuracy,
                E1_precision,
                E1_recall,
                E1_f1,
                E1_pr_auc,
                E1_fp,
                E1_fn,
                E1_tp
            ]
        })
    
        display(E0_E1_comparison)
    
        E0_E1_comparison.to_csv(
            RESULTS_DIR / "P1_E0_vs_E1_comparison.csv",
            index=False
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 66; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E1 Experiment Log with Actual Findings
        # ============================================================
    
        E1_log_row = pd.DataFrame([{
            "Experiment": "E1",
            "Change": "Introduced balanced class weights; architecture unchanged",
            "Hypothesis":
                "Increasing the training cost of fraud-class errors should "
                "improve fraud recall compared with the untreated E0 baseline.",
            "Observation":
                "Recall increased from 69.70% to 84.85% and false negatives "
                "fell from 20 to 10. However, false positives increased from "
                "6 to 123, reducing precision from 88.46% to 31.28%. "
                "F1 declined from 0.7797 to 0.4571 and PR-AUC declined "
                "from 0.8166 to 0.7296.",
            "Decision":
                "Do not select E1 as the current preferred configuration at "
                "the 0.50 threshold. The improvement in fraud detection was "
                "offset by excessive false-positive alerts. Proceed to SMOTE "
                "for comparison with an alternative imbalance strategy."
        }])
    
        experiment_log = pd.concat(
            [experiment_log, E1_log_row],
            ignore_index=True
        )
    
        experiment_log = experiment_log.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E1 experiment log saved successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 67; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Add E1 to Master Experiment Results
        # ============================================================
    
        E1_result_row = pd.DataFrame([{
            "Experiment": "E1",
            "Description": "Cost-Sensitive DNN",
            "Scaler": "StandardScaler",
            "Imbalance_Method": "Balanced Class Weights",
            "Architecture": "64-32-1",
            "Threshold": 0.50,
            "Accuracy": E1_accuracy,
            "Precision": E1_precision,
            "Recall": E1_recall,
            "F1": E1_f1,
            "PR_AUC": E1_pr_auc,
            "Average_Precision": E1_average_precision,
            "ROC_AUC": E1_roc_auc,
            "TN": E1_tn,
            "FP": E1_fp,
            "FN": E1_fn,
            "TP": E1_tp
        }])
    
        experiment_results = pd.read_csv(
            experiment_results_path
        )
    
        experiment_results = pd.concat(
            [experiment_results, E1_result_row],
            ignore_index=True
        )
    
        experiment_results = experiment_results.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E1 added to master results successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 68; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Housekeeping: Correct E0 Imbalance Method Label
        # ============================================================
    
        experiment_results.loc[
            experiment_results["Experiment"] == "E0",
            "Imbalance_Method"
        ] = "No imbalance treatment"
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E0 imbalance-method label corrected.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 69; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E1 Validation Predictions Permanently
        # ============================================================
    
        E1_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E1_val_probabilities,
            "Predicted_Class_0.50": E1_val_predictions
        })
    
        E1_predictions_df.to_csv(
            RESULTS_DIR / "P1_E1_validation_predictions.csv",
            index=False
        )
    
        print("E1 validation predictions saved successfully.")
else:
    print("Inspection mode: code not executed.")


**9. Experiment E2 – SMOTE DNN**

Experiment E2 evaluates Synthetic Minority Over-sampling Technique (SMOTE) as an alternative to cost-sensitive learning. SMOTE is applied exclusively to the standardised training subset to generate synthetic minority-class observations, while the validation and final test subsets remain unchanged. The baseline DNN architecture and training configuration are retained to support a controlled comparison with E0 and E1.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 71; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 9.1 SMOTE Setup
        # ============================================================
    
        from imblearn.over_sampling import SMOTE
    
        print("SMOTE imported successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 72; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 9.2 Train OR Restore Experiment E2 - SMOTE DNN
        # ============================================================
    
        E2_EPOCHS = 30
        E2_BATCH_SIZE = 1024
    
        E2_MODEL_PATH = MODEL_DIR / "P1_E2_smote_dnn.keras"
        E2_HISTORY_PATH = RESULTS_DIR / "P1_E2_training_history.csv"
        E2_SMOTE_SUMMARY_PATH = RESULTS_DIR / "P1_E2_smote_summary.csv"
    
        if (
            E2_MODEL_PATH.exists()
            and E2_HISTORY_PATH.exists()
            and E2_SMOTE_SUMMARY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E2 experiment found.")
            print("Loading saved E2 model and history instead of retraining...")
    
            E2_model = tf.keras.models.load_model(E2_MODEL_PATH)
    
            history_E2_df = pd.read_csv(
                E2_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E2 = SimpleNamespace(
                history=history_E2_df.to_dict(orient="list")
            )
    
            E2_smote_summary = pd.read_csv(
                E2_SMOTE_SUMMARY_PATH
            )
    
            display(E2_smote_summary)
    
            print("E2 restored successfully.")
    
        else:
            print("No saved E2 experiment found.")
            print("Applying SMOTE to TRAINING DATA ONLY...")
    
            smote = SMOTE(
                random_state=SEED,
                k_neighbors=5
            )
    
            X_train_smote, y_train_smote = smote.fit_resample(
                X_train_scaled,
                y_train
            )
    
            # Record class distribution before and after SMOTE
            before_legitimate = int((y_train == 0).sum())
            before_fraud = int((y_train == 1).sum())
    
            after_legitimate = int((y_train_smote == 0).sum())
            after_fraud = int((y_train_smote == 1).sum())
    
            E2_smote_summary = pd.DataFrame({
                "Stage": [
                    "Before SMOTE",
                    "After SMOTE"
                ],
                "Total Rows": [
                    len(y_train),
                    len(y_train_smote)
                ],
                "Legitimate": [
                    before_legitimate,
                    after_legitimate
                ],
                "Fraud": [
                    before_fraud,
                    after_fraud
                ],
                "Fraud Percentage": [
                    before_fraud / len(y_train) * 100,
                    after_fraud / len(y_train_smote) * 100
                ]
            })
    
            display(E2_smote_summary)
    
            E2_smote_summary.to_csv(
                E2_SMOTE_SUMMARY_PATH,
                index=False
            )
    
            print("\nBuilding fresh E2 model...")
    
            tf.keras.backend.clear_session()
    
            random.seed(SEED)
            np.random.seed(SEED)
            tf.random.set_seed(SEED)
    
            E2_model = Sequential([
                Input(shape=(X_train_smote.shape[1],)),
    
                Dense(
                    64,
                    activation="relu",
                    name="hidden_layer_1"
                ),
    
                Dense(
                    32,
                    activation="relu",
                    name="hidden_layer_2"
                ),
    
                Dense(
                    1,
                    activation="sigmoid",
                    name="fraud_probability"
                )
            ])
    
            E2_model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=0.001
                ),
                loss="binary_crossentropy",
                metrics=[
                    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                    tf.keras.metrics.Precision(name="precision"),
                    tf.keras.metrics.Recall(name="recall"),
                    tf.keras.metrics.AUC(
                        curve="PR",
                        name="pr_auc"
                    )
                ]
            )
    
            E2_early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=1
            )
    
            print("\nTraining E2...")
    
            history_E2 = E2_model.fit(
                X_train_smote,
                y_train_smote,
                validation_data=(
                    X_val_scaled,
                    y_val
                ),
                epochs=E2_EPOCHS,
                batch_size=E2_BATCH_SIZE,
                callbacks=[E2_early_stopping],
                verbose=1
            )
    
            # Save immediately
            E2_model.save(E2_MODEL_PATH)
    
            history_E2_df = pd.DataFrame(
                history_E2.history
            )
    
            history_E2_df.index.name = "Epoch"
    
            history_E2_df.to_csv(
                E2_HISTORY_PATH
            )
    
            print("\nE2 training completed and saved.")
    
            # Free the large synthetic arrays from RAM
            del X_train_smote
            del y_train_smote
    
        print("\nE2 model path:", E2_MODEL_PATH)
        print("E2 history path:", E2_HISTORY_PATH)
        print("E2 SMOTE summary path:", E2_SMOTE_SUMMARY_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 73; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 9.3 E2 Validation Predictions
        # ============================================================
    
        E2_val_probabilities = E2_model.predict(
            X_val_scaled,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E2_THRESHOLD = 0.50
    
        E2_val_predictions = (
            E2_val_probabilities >= E2_THRESHOLD
        ).astype(int)
    
        print("E2 validation predictions generated.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 74; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 9.4 E2 Validation Metrics
        # ============================================================
    
        E2_accuracy = accuracy_score(
            y_val,
            E2_val_predictions
        )
    
        E2_precision = precision_score(
            y_val,
            E2_val_predictions,
            zero_division=0
        )
    
        E2_recall = recall_score(
            y_val,
            E2_val_predictions,
            zero_division=0
        )
    
        E2_f1 = f1_score(
            y_val,
            E2_val_predictions,
            zero_division=0
        )
    
        precision_curve_E2, recall_curve_E2, thresholds_E2 = (
            precision_recall_curve(
                y_val,
                E2_val_probabilities
            )
        )
    
        E2_pr_auc = auc(
            recall_curve_E2,
            precision_curve_E2
        )
    
        E2_average_precision = average_precision_score(
            y_val,
            E2_val_probabilities
        )
    
        E2_roc_auc = roc_auc_score(
            y_val,
            E2_val_probabilities
        )
    
        E2_tn, E2_fp, E2_fn, E2_tp = confusion_matrix(
            y_val,
            E2_val_predictions
        ).ravel()
    
        E2_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E2_accuracy,
                E2_precision,
                E2_recall,
                E2_f1,
                E2_pr_auc,
                E2_average_precision,
                E2_roc_auc,
                E2_tn,
                E2_fp,
                E2_fn,
                E2_tp
            ]
        })
    
        display(E2_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 75; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E2 Validation Predictions Permanently
        # ============================================================
    
        E2_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E2_val_probabilities,
            "Predicted_Class_0.50": E2_val_predictions
        })
    
        E2_predictions_df.to_csv(
            RESULTS_DIR / "P1_E2_validation_predictions.csv",
            index=False
        )
    
        print("E2 validation predictions saved successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 76; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Add E2 to Master Experiment Results
        # ============================================================
    
        E2_result_row = pd.DataFrame([{
            "Experiment": "E2",
            "Description": "SMOTE DNN",
            "Scaler": "StandardScaler",
            "Imbalance_Method": "SMOTE",
            "Architecture": "64-32-1",
            "Threshold": 0.50,
            "Accuracy": E2_accuracy,
            "Precision": E2_precision,
            "Recall": E2_recall,
            "F1": E2_f1,
            "PR_AUC": E2_pr_auc,
            "Average_Precision": E2_average_precision,
            "ROC_AUC": E2_roc_auc,
            "TN": E2_tn,
            "FP": E2_fp,
            "FN": E2_fn,
            "TP": E2_tp
        }])
    
        experiment_results = pd.read_csv(
            experiment_results_path
        )
    
        experiment_results = pd.concat(
            [experiment_results, E2_result_row],
            ignore_index=True
        )
    
        experiment_results = experiment_results.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E2 added to master results successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 77; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # E0 vs E1 vs E2 Imbalance Strategy Comparison
        # ============================================================
    
        imbalance_comparison = experiment_results[
            experiment_results["Experiment"].isin(
                ["E0", "E1", "E2"]
            )
        ][[
            "Experiment",
            "Imbalance_Method",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "PR_AUC",
            "FP",
            "FN",
            "TP"
        ]].copy()
    
        display(imbalance_comparison)
    
        imbalance_comparison.to_csv(
            RESULTS_DIR / "P1_E0_E1_E2_imbalance_comparison.csv",
            index=False
        )
    
        print("Three-way imbalance comparison saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 78; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E2 Experiment Log with Actual Findings
        # ============================================================
    
        E2_log_row = pd.DataFrame([{
            "Experiment": "E2",
            "Change":
                "Applied SMOTE exclusively to the standardised training "
                "subset; DNN architecture unchanged.",
            "Hypothesis":
                "Synthetic minority oversampling should improve fraud recall "
                "while potentially producing a better precision-recall trade-off "
                "than balanced class weighting.",
            "Observation":
                "SMOTE increased fraud recall from 69.70% in E0 to 78.79% "
                "and reduced false negatives from 20 to 14. However, false "
                "positives increased from 6 to 50 and precision decreased from "
                "88.46% to 50.98%. E2 achieved F1=0.6190 and PR-AUC=0.7910. "
                "This was more balanced than E1 but remained inferior to E0 "
                "on both F1 and PR-AUC at the 0.50 threshold.",
            "Decision":
                "Retain E2 as evidence that SMOTE improves minority-class "
                "sensitivity but do not select it as the current preferred "
                "configuration. E0 remains the strongest overall baseline."
        }])
    
        experiment_log = pd.read_csv(experiment_log_path)
    
        experiment_log = pd.concat(
            [experiment_log, E2_log_row],
            ignore_index=True
        )
    
        experiment_log = experiment_log.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E2 experiment log saved successfully.")
else:
    print("Inspection mode: code not executed.")


**10. Experiment E3 – Scaling Comparison**

Experiment E3 evaluates RobustScaler as an alternative preprocessing strategy to the StandardScaler used in the E0 baseline. Exploratory analysis identified substantial positive skewness and extreme values in the transaction amount feature. RobustScaler is therefore investigated because its use of median and interquartile-range statistics makes it less sensitive to extreme observations. All other major modelling choices are retained from E0 to isolate the effect of the scaling method.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 80; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 10.1 RobustScaler Preprocessing
        # ============================================================
    
        from sklearn.preprocessing import RobustScaler
        import joblib
    
        ROBUST_SCALER_PATH = MODEL_DIR / "P1_robust_scaler.joblib"
    
        if (
            ROBUST_SCALER_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing RobustScaler found.")
            print("Loading saved RobustScaler...")
    
            robust_scaler = joblib.load(
                ROBUST_SCALER_PATH
            )
    
        else:
            print("Fitting RobustScaler on TRAINING DATA ONLY...")
    
            robust_scaler = RobustScaler()
    
            robust_scaler.fit(X_train)
    
            joblib.dump(
                robust_scaler,
                ROBUST_SCALER_PATH
            )
    
            print("RobustScaler fitted and saved.")
    
        # Apply training-derived transformation
        X_train_robust = robust_scaler.transform(X_train)
        X_val_robust = robust_scaler.transform(X_val)
        X_test_robust = robust_scaler.transform(X_test)
    
        print("\nRobust-scaled shapes:")
        print("Training:", X_train_robust.shape)
        print("Validation:", X_val_robust.shape)
        print("Test:", X_test_robust.shape)
    
        print("\nNaN values:", np.isnan(X_train_robust).sum())
        print("Infinite values:", np.isinf(X_train_robust).sum())
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 81; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 10.2 Train OR Restore Experiment E3 - RobustScaler DNN
        # ============================================================
    
        E3_EPOCHS = 30
        E3_BATCH_SIZE = 1024
    
        E3_MODEL_PATH = MODEL_DIR / "P1_E3_robust_scaler_dnn.keras"
        E3_HISTORY_PATH = RESULTS_DIR / "P1_E3_training_history.csv"
    
        if (
            E3_MODEL_PATH.exists()
            and E3_HISTORY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E3 experiment found.")
            print("Loading saved E3 model and history instead of retraining...")
    
            E3_model = tf.keras.models.load_model(
                E3_MODEL_PATH
            )
    
            history_E3_df = pd.read_csv(
                E3_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E3 = SimpleNamespace(
                history=history_E3_df.to_dict(orient="list")
            )
    
            print("E3 restored successfully.")
    
        else:
            print("No saved E3 experiment found.")
            print("Building fresh RobustScaler DNN...")
    
            tf.keras.backend.clear_session()
    
            random.seed(SEED)
            np.random.seed(SEED)
            tf.random.set_seed(SEED)
    
            E3_model = Sequential([
                Input(shape=(X_train_robust.shape[1],)),
    
                Dense(
                    64,
                    activation="relu",
                    name="hidden_layer_1"
                ),
    
                Dense(
                    32,
                    activation="relu",
                    name="hidden_layer_2"
                ),
    
                Dense(
                    1,
                    activation="sigmoid",
                    name="fraud_probability"
                )
            ])
    
            E3_model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=0.001
                ),
                loss="binary_crossentropy",
                metrics=[
                    tf.keras.metrics.BinaryAccuracy(name="accuracy"),
                    tf.keras.metrics.Precision(name="precision"),
                    tf.keras.metrics.Recall(name="recall"),
                    tf.keras.metrics.AUC(
                        curve="PR",
                        name="pr_auc"
                    )
                ]
            )
    
            E3_early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=1
            )
    
            print("Training E3...")
    
            history_E3 = E3_model.fit(
                X_train_robust,
                y_train,
                validation_data=(
                    X_val_robust,
                    y_val
                ),
                epochs=E3_EPOCHS,
                batch_size=E3_BATCH_SIZE,
                callbacks=[E3_early_stopping],
                verbose=1
            )
    
            E3_model.save(
                E3_MODEL_PATH
            )
    
            history_E3_df = pd.DataFrame(
                history_E3.history
            )
    
            history_E3_df.index.name = "Epoch"
    
            history_E3_df.to_csv(
                E3_HISTORY_PATH
            )
    
            print("E3 training completed and saved.")
    
        print("\nE3 model path:", E3_MODEL_PATH)
        print("E3 history path:", E3_HISTORY_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 82; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 10.3 E3 Validation Predictions
        # ============================================================
    
        E3_val_probabilities = E3_model.predict(
            X_val_robust,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E3_THRESHOLD = 0.50
    
        E3_val_predictions = (
            E3_val_probabilities >= E3_THRESHOLD
        ).astype(int)
    
        print("E3 validation predictions generated.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 83; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 10.4 E3 Validation Metrics
        # ============================================================
    
        E3_accuracy = accuracy_score(
            y_val,
            E3_val_predictions
        )
    
        E3_precision = precision_score(
            y_val,
            E3_val_predictions,
            zero_division=0
        )
    
        E3_recall = recall_score(
            y_val,
            E3_val_predictions,
            zero_division=0
        )
    
        E3_f1 = f1_score(
            y_val,
            E3_val_predictions,
            zero_division=0
        )
    
        precision_curve_E3, recall_curve_E3, thresholds_E3 = (
            precision_recall_curve(
                y_val,
                E3_val_probabilities
            )
        )
    
        E3_pr_auc = auc(
            recall_curve_E3,
            precision_curve_E3
        )
    
        E3_average_precision = average_precision_score(
            y_val,
            E3_val_probabilities
        )
    
        E3_roc_auc = roc_auc_score(
            y_val,
            E3_val_probabilities
        )
    
        E3_tn, E3_fp, E3_fn, E3_tp = confusion_matrix(
            y_val,
            E3_val_predictions
        ).ravel()
    
        E3_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E3_accuracy,
                E3_precision,
                E3_recall,
                E3_f1,
                E3_pr_auc,
                E3_average_precision,
                E3_roc_auc,
                E3_tn,
                E3_fp,
                E3_fn,
                E3_tp
            ]
        })
    
        display(E3_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 84; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E3 Validation Predictions Permanently
        # ============================================================
    
        E3_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E3_val_probabilities,
            "Predicted_Class_0.50": E3_val_predictions
        })
    
        E3_predictions_df.to_csv(
            RESULTS_DIR / "P1_E3_validation_predictions.csv",
            index=False
        )
    
        print("E3 validation predictions saved successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 85; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Add E3 to Master Experiment Results
        # ============================================================
    
        E3_result_row = pd.DataFrame([{
            "Experiment": "E3",
            "Description": "RobustScaler DNN",
            "Scaler": "RobustScaler",
            "Imbalance_Method": "No imbalance treatment",
            "Architecture": "64-32-1",
            "Threshold": 0.50,
            "Accuracy": E3_accuracy,
            "Precision": E3_precision,
            "Recall": E3_recall,
            "F1": E3_f1,
            "PR_AUC": E3_pr_auc,
            "Average_Precision": E3_average_precision,
            "ROC_AUC": E3_roc_auc,
            "TN": E3_tn,
            "FP": E3_fp,
            "FN": E3_fn,
            "TP": E3_tp
        }])
    
        experiment_results = pd.read_csv(
            experiment_results_path
        )
    
        experiment_results = pd.concat(
            [experiment_results, E3_result_row],
            ignore_index=True
        )
    
        experiment_results = experiment_results.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E3 added to master results successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 86; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # E0 vs E3 Scaling Comparison
        # ============================================================
    
        scaling_comparison = experiment_results[
            experiment_results["Experiment"].isin(
                ["E0", "E3"]
            )
        ][[
            "Experiment",
            "Scaler",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "PR_AUC",
            "FP",
            "FN",
            "TP"
        ]].copy()
    
        display(scaling_comparison)
    
        scaling_comparison.to_csv(
            RESULTS_DIR / "P1_E0_E3_scaling_comparison.csv",
            index=False
        )
    
        print("Scaling comparison saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 87; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E3 Experiment Log with Actual Findings
        # ============================================================
    
        E3_log_row = pd.DataFrame([{
            "Experiment": "E3",
            "Change":
                "Replaced StandardScaler with RobustScaler while retaining "
                "the E0 architecture and no imbalance treatment.",
            "Hypothesis":
                "RobustScaler may improve performance because the transaction "
                "amount feature contains substantial skewness and extreme values.",
            "Observation":
                "RobustScaler produced modest but consistent improvements over E0. "
                "Precision increased from 88.46% to 92.16%, recall from 69.70% "
                "to 71.21%, F1 from 0.7797 to 0.8034 and PR-AUC from 0.8166 "
                "to 0.8186. False positives decreased from 6 to 4 and false "
                "negatives from 20 to 19.",
            "Decision":
                "Select RobustScaler as the preferred preprocessing method for "
                "subsequent experiments because it improved the overall "
                "precision-recall balance without increasing classification errors."
        }])
    
        experiment_log = pd.read_csv(experiment_log_path)
    
        experiment_log = pd.concat(
            [experiment_log, E3_log_row],
            ignore_index=True
        )
    
        experiment_log = experiment_log.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E3 experiment log saved successfully.")
else:
    print("Inspection mode: code not executed.")


**11. Experiment E4 – Improved DNN Architecture**

Experiment E4 investigates whether increased network capacity combined with Batch Normalisation and Dropout improves generalisation over the simpler E3 architecture. RobustScaler and the absence of class-imbalance correction are retained from E3. The architectural changes therefore represent the primary experimental intervention.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 89; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 11.1 E4 Architecture Components
        # ============================================================
    
        from tensorflow.keras.layers import (
            Input,
            Dense,
            BatchNormalization,
            Activation,
            Dropout
        )
    
        print("E4 architecture components imported successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 90; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 11.2 Train OR Restore Experiment E4
        # Improved Architecture
        # ============================================================
    
        E4_EPOCHS = 30
        E4_BATCH_SIZE = 1024
    
        E4_MODEL_PATH = MODEL_DIR / "P1_E4_improved_architecture.keras"
        E4_HISTORY_PATH = RESULTS_DIR / "P1_E4_training_history.csv"
    
        if (
            E4_MODEL_PATH.exists()
            and E4_HISTORY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E4 experiment found.")
            print("Loading saved E4 model and history instead of retraining...")
    
            E4_model = tf.keras.models.load_model(
                E4_MODEL_PATH
            )
    
            history_E4_df = pd.read_csv(
                E4_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E4 = SimpleNamespace(
                history=history_E4_df.to_dict(orient="list")
            )
    
            print("E4 restored successfully.")
    
        else:
            print("No saved E4 experiment found.")
            print("Building improved DNN architecture...")
    
            tf.keras.backend.clear_session()
    
            random.seed(SEED)
            np.random.seed(SEED)
            tf.random.set_seed(SEED)
    
            E4_model = Sequential([
                Input(shape=(X_train_robust.shape[1],)),
    
                Dense(
                    128,
                    name="dense_128"
                ),
                BatchNormalization(
                    name="batchnorm_1"
                ),
                Activation(
                    "relu",
                    name="relu_1"
                ),
                Dropout(
                    0.30,
                    name="dropout_30"
                ),
    
                Dense(
                    64,
                    name="dense_64"
                ),
                BatchNormalization(
                    name="batchnorm_2"
                ),
                Activation(
                    "relu",
                    name="relu_2"
                ),
                Dropout(
                    0.20,
                    name="dropout_20"
                ),
    
                Dense(
                    32,
                    activation="relu",
                    name="dense_32"
                ),
    
                Dense(
                    1,
                    activation="sigmoid",
                    name="fraud_probability"
                )
            ])
    
            E4_model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=0.001
                ),
                loss="binary_crossentropy",
                metrics=[
                    tf.keras.metrics.BinaryAccuracy(
                        name="accuracy"
                    ),
                    tf.keras.metrics.Precision(
                        name="precision"
                    ),
                    tf.keras.metrics.Recall(
                        name="recall"
                    ),
                    tf.keras.metrics.AUC(
                        curve="PR",
                        name="pr_auc"
                    )
                ]
            )
    
            E4_model.summary()
    
            E4_early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=1
            )
    
            print("\nTraining E4...")
    
            history_E4 = E4_model.fit(
                X_train_robust,
                y_train,
                validation_data=(
                    X_val_robust,
                    y_val
                ),
                epochs=E4_EPOCHS,
                batch_size=E4_BATCH_SIZE,
                callbacks=[E4_early_stopping],
                verbose=1
            )
    
            # Save immediately
            E4_model.save(
                E4_MODEL_PATH
            )
    
            history_E4_df = pd.DataFrame(
                history_E4.history
            )
    
            history_E4_df.index.name = "Epoch"
    
            history_E4_df.to_csv(
                E4_HISTORY_PATH
            )
    
            print("\nE4 training completed and saved.")
    
        print("\nE4 model path:", E4_MODEL_PATH)
        print("E4 history path:", E4_HISTORY_PATH)
    
        print("\nE4 model architecture:")
        E4_model.summary()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 91; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 11.3 E4 Validation Predictions
        # ============================================================
    
        E4_val_probabilities = E4_model.predict(
            X_val_robust,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E4_THRESHOLD = 0.50
    
        E4_val_predictions = (
            E4_val_probabilities >= E4_THRESHOLD
        ).astype(int)
    
        print("E4 validation predictions generated.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 92; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 11.4 E4 Validation Metrics
        # ============================================================
    
        E4_accuracy = accuracy_score(
            y_val,
            E4_val_predictions
        )
    
        E4_precision = precision_score(
            y_val,
            E4_val_predictions,
            zero_division=0
        )
    
        E4_recall = recall_score(
            y_val,
            E4_val_predictions,
            zero_division=0
        )
    
        E4_f1 = f1_score(
            y_val,
            E4_val_predictions,
            zero_division=0
        )
    
        precision_curve_E4, recall_curve_E4, thresholds_E4 = (
            precision_recall_curve(
                y_val,
                E4_val_probabilities
            )
        )
    
        E4_pr_auc = auc(
            recall_curve_E4,
            precision_curve_E4
        )
    
        E4_average_precision = average_precision_score(
            y_val,
            E4_val_probabilities
        )
    
        E4_roc_auc = roc_auc_score(
            y_val,
            E4_val_probabilities
        )
    
        E4_tn, E4_fp, E4_fn, E4_tp = confusion_matrix(
            y_val,
            E4_val_predictions
        ).ravel()
    
        E4_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E4_accuracy,
                E4_precision,
                E4_recall,
                E4_f1,
                E4_pr_auc,
                E4_average_precision,
                E4_roc_auc,
                E4_tn,
                E4_fp,
                E4_fn,
                E4_tp
            ]
        })
    
        display(E4_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 93; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E4 Validation Predictions Permanently
        # ============================================================
    
        E4_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E4_val_probabilities,
            "Predicted_Class_0.50": E4_val_predictions
        })
    
        E4_predictions_df.to_csv(
            RESULTS_DIR / "P1_E4_validation_predictions.csv",
            index=False
        )
    
        print("E4 validation predictions saved successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 94; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Add E4 to Master Experiment Results
        # ============================================================
    
        E4_result_row = pd.DataFrame([{
            "Experiment": "E4",
            "Description": "Improved Architecture DNN",
            "Scaler": "RobustScaler",
            "Imbalance_Method": "No imbalance treatment",
            "Architecture": "128-BN-D0.30-64-BN-D0.20-32-1",
            "Threshold": 0.50,
            "Accuracy": E4_accuracy,
            "Precision": E4_precision,
            "Recall": E4_recall,
            "F1": E4_f1,
            "PR_AUC": E4_pr_auc,
            "Average_Precision": E4_average_precision,
            "ROC_AUC": E4_roc_auc,
            "TN": E4_tn,
            "FP": E4_fp,
            "FN": E4_fn,
            "TP": E4_tp
        }])
    
        experiment_results = pd.read_csv(
            experiment_results_path
        )
    
        experiment_results = pd.concat(
            [experiment_results, E4_result_row],
            ignore_index=True
        )
    
        experiment_results = experiment_results.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E4 added to master results successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 95; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # E3 vs E4 Architecture Comparison
        # ============================================================
    
        architecture_comparison = experiment_results[
            experiment_results["Experiment"].isin(
                ["E3", "E4"]
            )
        ][[
            "Experiment",
            "Architecture",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "PR_AUC",
            "FP",
            "FN",
            "TP"
        ]].copy()
    
        display(architecture_comparison)
    
        architecture_comparison.to_csv(
            RESULTS_DIR / "P1_E3_E4_architecture_comparison.csv",
            index=False
        )
    
        print("Architecture comparison saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 96; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E4 Experiment Log with Actual Findings
        # ============================================================
    
        E4_log_row = pd.DataFrame([{
            "Experiment": "E4",
            "Change":
                "Expanded the DNN to 128-64-32 hidden units and introduced "
                "Batch Normalisation and Dropout while retaining RobustScaler.",
            "Hypothesis":
                "Greater network capacity combined with regularisation may "
                "improve minority-class generalisation and PR-AUC.",
            "Observation":
                "E4 increased recall from 71.21% to 75.76%, reduced false "
                "negatives from 19 to 16 and increased PR-AUC from 0.8186 "
                "to 0.8343. F1 increased slightly from 0.8034 to 0.8065. "
                "However, precision decreased from 92.16% to 86.21% and "
                "false positives increased from 4 to 8.",
            "Decision":
                "Retain E4 as the current preferred architecture because it "
                "achieved the strongest PR-AUC so far and improved fraud recall. "
                "The precision-recall trade-off will be reconsidered during "
                "decision-threshold optimisation."
        }])
    
        experiment_log = pd.read_csv(experiment_log_path)
    
        experiment_log = pd.concat(
            [experiment_log, E4_log_row],
            ignore_index=True
        )
    
        experiment_log = experiment_log.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(experiment_log)
    
        print("E4 experiment log saved successfully.")
else:
    print("Inspection mode: code not executed.")


**12. Experiment E5 – Learning Rate and Training-Schedule Optimisation**

Experiment E5 retained the E4 architecture, RobustScaler preprocessing and initial Adam learning rate of 0.001, while refining the training schedule. `ReduceLROnPlateau` was introduced to reduce the learning rate when validation loss stopped improving. The maximum training duration was increased from 30 to 40 epochs and EarlyStopping patience was increased from 5 to 7 epochs, allowing the model additional opportunity to benefit from reduced learning rates before training terminated.

This experiment therefore evaluates the combined effect of adaptive learning-rate reduction and a more flexible training schedule rather than attributing any performance change solely to learning-rate adjustment.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 98; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 12.1 Learning Rate Optimisation Components
        # ============================================================
    
        from tensorflow.keras.callbacks import ReduceLROnPlateau
    
        print("E5 learning-rate callback imported successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 99; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 12.2 Train OR Restore Experiment E5
        # Adaptive Learning Rate
        # ============================================================
    
        E5_EPOCHS = 40
        E5_BATCH_SIZE = 1024
    
        E5_MODEL_PATH = MODEL_DIR / "P1_E5_lr_optimised_dnn.keras"
        E5_HISTORY_PATH = RESULTS_DIR / "P1_E5_training_history.csv"
    
        if (
            E5_MODEL_PATH.exists()
            and E5_HISTORY_PATH.exists()
            and not FORCE_RETRAIN
        ):
            print("Existing E5 experiment found.")
            print("Loading saved E5 model and history instead of retraining...")
    
            E5_model = tf.keras.models.load_model(
                E5_MODEL_PATH
            )
    
            history_E5_df = pd.read_csv(
                E5_HISTORY_PATH,
                index_col="Epoch"
            )
    
            history_E5 = SimpleNamespace(
                history=history_E5_df.to_dict(orient="list")
            )
    
            print("E5 restored successfully.")
    
        else:
            print("No saved E5 experiment found.")
            print("Building E5 using the improved E4 architecture...")
    
            tf.keras.backend.clear_session()
    
            random.seed(SEED)
            np.random.seed(SEED)
            tf.random.set_seed(SEED)
    
            E5_model = Sequential([
                Input(shape=(X_train_robust.shape[1],)),
    
                Dense(
                    128,
                    name="dense_128"
                ),
                BatchNormalization(
                    name="batchnorm_1"
                ),
                Activation(
                    "relu",
                    name="relu_1"
                ),
                Dropout(
                    0.30,
                    name="dropout_30"
                ),
    
                Dense(
                    64,
                    name="dense_64"
                ),
                BatchNormalization(
                    name="batchnorm_2"
                ),
                Activation(
                    "relu",
                    name="relu_2"
                ),
                Dropout(
                    0.20,
                    name="dropout_20"
                ),
    
                Dense(
                    32,
                    activation="relu",
                    name="dense_32"
                ),
    
                Dense(
                    1,
                    activation="sigmoid",
                    name="fraud_probability"
                )
            ])
    
            E5_model.compile(
                optimizer=tf.keras.optimizers.Adam(
                    learning_rate=0.001
                ),
                loss="binary_crossentropy",
                metrics=[
                    tf.keras.metrics.BinaryAccuracy(
                        name="accuracy"
                    ),
                    tf.keras.metrics.Precision(
                        name="precision"
                    ),
                    tf.keras.metrics.Recall(
                        name="recall"
                    ),
                    tf.keras.metrics.AUC(
                        curve="PR",
                        name="pr_auc"
                    )
                ]
            )
    
            E5_early_stopping = EarlyStopping(
                monitor="val_loss",
                patience=7,
                restore_best_weights=True,
                verbose=1
            )
    
            E5_reduce_lr = ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=3,
                min_lr=0.00001,
                verbose=1
            )
    
            print("\nTraining E5...")
    
            history_E5 = E5_model.fit(
                X_train_robust,
                y_train,
                validation_data=(
                    X_val_robust,
                    y_val
                ),
                epochs=E5_EPOCHS,
                batch_size=E5_BATCH_SIZE,
                callbacks=[
                    E5_reduce_lr,
                    E5_early_stopping
                ],
                verbose=1
            )
    
            E5_model.save(
                E5_MODEL_PATH
            )
    
            history_E5_df = pd.DataFrame(
                history_E5.history
            )
    
            history_E5_df.index.name = "Epoch"
    
            history_E5_df.to_csv(
                E5_HISTORY_PATH
            )
    
            print("\nE5 training completed and saved.")
    
        print("\nE5 model path:", E5_MODEL_PATH)
        print("E5 history path:", E5_HISTORY_PATH)
    
        print("\nE5 model architecture:")
        E5_model.summary()
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 100; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 12.3 E5 Validation Predictions
        # ============================================================
    
        E5_val_probabilities = E5_model.predict(
            X_val_robust,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E5_THRESHOLD = 0.50
    
        E5_val_predictions = (
            E5_val_probabilities >= E5_THRESHOLD
        ).astype(int)
    
        print("E5 validation predictions generated.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 101; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 12.4 E5 Validation Metrics
        # ============================================================
    
        E5_accuracy = accuracy_score(
            y_val,
            E5_val_predictions
        )
    
        E5_precision = precision_score(
            y_val,
            E5_val_predictions,
            zero_division=0
        )
    
        E5_recall = recall_score(
            y_val,
            E5_val_predictions,
            zero_division=0
        )
    
        E5_f1 = f1_score(
            y_val,
            E5_val_predictions,
            zero_division=0
        )
    
        precision_curve_E5, recall_curve_E5, thresholds_E5 = (
            precision_recall_curve(
                y_val,
                E5_val_probabilities
            )
        )
    
        E5_pr_auc = auc(
            recall_curve_E5,
            precision_curve_E5
        )
    
        E5_average_precision = average_precision_score(
            y_val,
            E5_val_probabilities
        )
    
        E5_roc_auc = roc_auc_score(
            y_val,
            E5_val_probabilities
        )
    
        E5_tn, E5_fp, E5_fn, E5_tp = confusion_matrix(
            y_val,
            E5_val_predictions
        ).ravel()
    
        E5_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E5_accuracy,
                E5_precision,
                E5_recall,
                E5_f1,
                E5_pr_auc,
                E5_average_precision,
                E5_roc_auc,
                E5_tn,
                E5_fp,
                E5_fn,
                E5_tp
            ]
        })
    
        display(E5_metrics)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 102; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save E5 Validation Predictions Permanently
        # ============================================================
    
        E5_predictions_df = pd.DataFrame({
            "Actual_Class": y_val.to_numpy(),
            "Fraud_Probability": E5_val_probabilities,
            "Predicted_Class_0.50": E5_val_predictions
        })
    
        E5_predictions_df.to_csv(
            RESULTS_DIR / "P1_E5_validation_predictions.csv",
            index=False
        )
    
        print("E5 validation predictions saved successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 103; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Add E5 to Master Experiment Results
        # ============================================================
    
        E5_result_row = pd.DataFrame([{
            "Experiment": "E5",
            "Description": "Adaptive LR Optimised DNN",
            "Scaler": "RobustScaler",
            "Imbalance_Method": "No imbalance treatment",
            "Architecture": "128-BN-D0.30-64-BN-D0.20-32-1",
            "Threshold": 0.50,
            "Accuracy": E5_accuracy,
            "Precision": E5_precision,
            "Recall": E5_recall,
            "F1": E5_f1,
            "PR_AUC": E5_pr_auc,
            "Average_Precision": E5_average_precision,
            "ROC_AUC": E5_roc_auc,
            "TN": E5_tn,
            "FP": E5_fp,
            "FN": E5_fn,
            "TP": E5_tp
        }])
    
        experiment_results = pd.read_csv(
            experiment_results_path
        )
    
        experiment_results = pd.concat(
            [experiment_results, E5_result_row],
            ignore_index=True
        )
    
        experiment_results = experiment_results.drop_duplicates(
            subset="Experiment",
            keep="last"
        )
    
        experiment_results.to_csv(
            experiment_results_path,
            index=False
        )
    
        display(experiment_results)
    
        print("E5 added to master results successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 104; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # E4 vs E5 Optimisation Comparison
        # ============================================================
    
        optimisation_comparison = experiment_results[
            experiment_results["Experiment"].isin(
                ["E4", "E5"]
            )
        ][[
            "Experiment",
            "Description",
            "Accuracy",
            "Precision",
            "Recall",
            "F1",
            "PR_AUC",
            "FP",
            "FN",
            "TP"
        ]].copy()
    
        display(optimisation_comparison)
    
        optimisation_comparison.to_csv(
            RESULTS_DIR / "P1_E4_E5_optimisation_comparison.csv",
            index=False
        )
    
        print("Optimisation comparison saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 105; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Update E5 Experiment Log with Actual Findings
        # ============================================================
    
        E5_log_row = pd.DataFrame([{
            "Experiment": "E5",
    
            "Change":
                "Retained the E4 architecture and RobustScaler while introducing "
                "ReduceLROnPlateau, increasing the maximum training duration "
                "from 30 to 40 epochs and increasing EarlyStopping patience "
                "from 5 to 7 epochs.",
    
            "Hypothesis":
                "Adaptive learning-rate reduction combined with a more flexible "
                "training schedule may allow finer optimisation after validation "
                "loss plateaus and improve generalisation.",
    
            "Observation":
                "E5 improved precision from 86.21% to 87.93%, recall from "
                "75.76% to 77.27%, F1 from 0.8065 to 0.8226 and PR-AUC "
                "from 0.8343 to 0.8432. False positives decreased from 8 "
                "to 7 and false negatives from 16 to 15. Training stopped "
                "at epoch 38 and restored weights from epoch 31.",
    
            "Decision":
                "Select E5 as the final candidate configuration before "
                "decision-threshold optimisation because the combined adaptive "
                "learning-rate and training-schedule refinement achieved the "
                "strongest overall validation precision-recall performance."
        }])
    
        experiment_log = pd.read_csv(
            experiment_log_path
        )
    
        experiment_log = pd.concat(
            [
                experiment_log,
                E5_log_row
            ],
            ignore_index=True
        )
    
        experiment_log = (
            experiment_log
            .drop_duplicates(
                subset="Experiment",
                keep="last"
            )
            .reset_index(drop=True)
        )
    
        experiment_log.to_csv(
            experiment_log_path,
            index=False
        )
    
        display(
            experiment_log
        )
    
        print(
            "E5 experiment log updated successfully."
        )
else:
    print("Inspection mode: code not executed.")


**13. Decision Threshold Analysis**

The E5 model produces continuous fraud probabilities that must be converted into binary decisions using a classification threshold. Although 0.50 was used consistently during controlled model comparisons, this default value may not provide the most appropriate precision-recall trade-off for an extremely imbalanced fraud-detection problem. Threshold selection is therefore conducted exclusively on the validation subset, while the held-out test set is not used for model or threshold selection and is reserved for final evaluation.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 107; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.1 Threshold Grid Analysis
        # ============================================================
    
        threshold_grid = np.arange(
            0.05,
            0.951,
            0.05
        )
    
        threshold_rows = []
    
        for threshold in threshold_grid:
    
            predictions = (
                E5_val_probabilities >= threshold
            ).astype(int)
    
            tn, fp, fn, tp = confusion_matrix(
                y_val,
                predictions
            ).ravel()
    
            precision = precision_score(
                y_val,
                predictions,
                zero_division=0
            )
    
            recall = recall_score(
                y_val,
                predictions,
                zero_division=0
            )
    
            f1 = f1_score(
                y_val,
                predictions,
                zero_division=0
            )
    
            threshold_rows.append({
                "Threshold": threshold,
                "Precision": precision,
                "Recall": recall,
                "F1": f1,
                "False_Positive": fp,
                "False_Negative": fn,
                "True_Positive": tp,
                "Predicted_Fraud": int(predictions.sum())
            })
    
        threshold_analysis = pd.DataFrame(
            threshold_rows
        )
    
        display(threshold_analysis)
    
        threshold_analysis.to_csv(
            RESULTS_DIR / "P1_E5_threshold_analysis.csv",
            index=False
        )
    
        print("Threshold grid analysis saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 108; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.2 Exact Validation F1-Optimal Threshold
        # ============================================================
    
        precision_values, recall_values, threshold_values = (
            precision_recall_curve(
                y_val,
                E5_val_probabilities
            )
        )
    
        # precision/recall contain one extra terminal value
        precision_for_thresholds = precision_values[:-1]
        recall_for_thresholds = recall_values[:-1]
    
        f1_values = (
            2
            * precision_for_thresholds
            * recall_for_thresholds
            / (
                precision_for_thresholds
                + recall_for_thresholds
                + 1e-12
            )
        )
    
        best_index = np.argmax(
            f1_values
        )
    
        E5_OPTIMAL_THRESHOLD = float(
            threshold_values[best_index]
        )
    
        optimal_precision = float(
            precision_for_thresholds[best_index]
        )
    
        optimal_recall = float(
            recall_for_thresholds[best_index]
        )
    
        optimal_f1 = float(
            f1_values[best_index]
        )
    
        print(
            f"Optimal validation threshold: "
            f"{E5_OPTIMAL_THRESHOLD:.6f}"
        )
    
        print(
            f"Precision at optimal threshold: "
            f"{optimal_precision:.6f}"
        )
    
        print(
            f"Recall at optimal threshold: "
            f"{optimal_recall:.6f}"
        )
    
        print(
            f"F1 at optimal threshold: "
            f"{optimal_f1:.6f}"
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 109; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.3 Evaluate Selected Validation Threshold
        # ============================================================
    
        E5_optimal_val_predictions = (
            E5_val_probabilities
            >= E5_OPTIMAL_THRESHOLD
        ).astype(int)
    
        optimal_accuracy = accuracy_score(
            y_val,
            E5_optimal_val_predictions
        )
    
        optimal_precision = precision_score(
            y_val,
            E5_optimal_val_predictions,
            zero_division=0
        )
    
        optimal_recall = recall_score(
            y_val,
            E5_optimal_val_predictions,
            zero_division=0
        )
    
        optimal_f1 = f1_score(
            y_val,
            E5_optimal_val_predictions,
            zero_division=0
        )
    
        optimal_tn, optimal_fp, optimal_fn, optimal_tp = (
            confusion_matrix(
                y_val,
                E5_optimal_val_predictions
            ).ravel()
        )
    
        selected_threshold_summary = pd.DataFrame({
            "Metric": [
                "Selected Threshold",
                "Accuracy",
                "Precision",
                "Recall",
                "F1",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                E5_OPTIMAL_THRESHOLD,
                optimal_accuracy,
                optimal_precision,
                optimal_recall,
                optimal_f1,
                optimal_tn,
                optimal_fp,
                optimal_fn,
                optimal_tp
            ]
        })
    
        display(
            selected_threshold_summary
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 110; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.4 Default vs Optimised Threshold
        # ============================================================
    
        threshold_comparison = pd.DataFrame({
            "Threshold Strategy": [
                "Default 0.50",
                "F1-Optimised"
            ],
            "Threshold": [
                0.50,
                E5_OPTIMAL_THRESHOLD
            ],
            "Precision": [
                E5_precision,
                optimal_precision
            ],
            "Recall": [
                E5_recall,
                optimal_recall
            ],
            "F1": [
                E5_f1,
                optimal_f1
            ],
            "False Positives": [
                E5_fp,
                optimal_fp
            ],
            "False Negatives": [
                E5_fn,
                optimal_fn
            ],
            "True Positives": [
                E5_tp,
                optimal_tp
            ]
        })
    
        display(
            threshold_comparison
        )
    
        threshold_comparison.to_csv(
            RESULTS_DIR /
            "P1_E5_threshold_comparison.csv",
            index=False
        )
    
        print(
            "Default vs optimised threshold comparison saved."
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 111; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # Save Selected Threshold Permanently
        # ============================================================
    
        selected_threshold_df = pd.DataFrame([{
            "Model": "E5",
            "Selection_Data": "Validation",
            "Selection_Criterion": "Maximum F1",
            "Threshold": E5_OPTIMAL_THRESHOLD,
            "Precision": optimal_precision,
            "Recall": optimal_recall,
            "F1": optimal_f1,
            "FP": optimal_fp,
            "FN": optimal_fn,
            "TP": optimal_tp
        }])
    
        SELECTED_THRESHOLD_PATH = (
            RESULTS_DIR /
            "P1_E5_selected_threshold.csv"
        )
    
        selected_threshold_df.to_csv(
            SELECTED_THRESHOLD_PATH,
            index=False
        )
    
        display(
            selected_threshold_df
        )
    
        print(
            "Selected threshold saved permanently to:"
        )
    
        print(
            SELECTED_THRESHOLD_PATH
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 112; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.5 Threshold Trade-Off Figure
        # ============================================================
    
        fig, ax = plt.subplots(
            figsize=(8, 5)
        )
    
        ax.plot(
            threshold_analysis["Threshold"],
            threshold_analysis["Precision"],
            label="Precision"
        )
    
        ax.plot(
            threshold_analysis["Threshold"],
            threshold_analysis["Recall"],
            label="Recall"
        )
    
        ax.plot(
            threshold_analysis["Threshold"],
            threshold_analysis["F1"],
            label="F1 Score"
        )
    
        ax.axvline(
            E5_OPTIMAL_THRESHOLD,
            linestyle="--",
            label=(
                f"Selected Threshold "
                f"({E5_OPTIMAL_THRESHOLD:.3f})"
            )
        )
    
        ax.set_title(
            "E5 Validation Performance Across Decision Thresholds"
        )
    
        ax.set_xlabel(
            "Fraud Decision Threshold"
        )
    
        ax.set_ylabel(
            "Metric Score"
        )
    
        ax.set_ylim(
            0,
            1.05
        )
    
        ax.legend()
    
        plt.tight_layout()
    
        THRESHOLD_FIGURE_PATH = (
            FIGURE_DIR /
            "P1_E5_threshold_tradeoff.png"
        )
    
        plt.savefig(
            THRESHOLD_FIGURE_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print(
            "Threshold figure saved to:"
        )
    
        print(
            THRESHOLD_FIGURE_PATH
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 113; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 13.6 Lock Final Project 1 Model Configuration
        # ============================================================
    
        FINAL_MODEL_NAME = "E5"
        FINAL_THRESHOLD = E5_OPTIMAL_THRESHOLD
    
        final_configuration = pd.DataFrame([{
            "Final_Model": "E5 Adaptive LR Optimised DNN",
            "Model_File": "P1_E5_lr_optimised_dnn.keras",
            "Scaler": "RobustScaler",
            "Scaler_File": "P1_robust_scaler.joblib",
            "Imbalance_Method": "No imbalance treatment",
            "Architecture": "128-BN-D0.30-64-BN-D0.20-32-1",
            "Optimizer": "Adam + ReduceLROnPlateau",
            "Threshold": FINAL_THRESHOLD,
            "Threshold_Selected_On": "Validation set",
            "Threshold_Criterion": "Maximum F1",
            "Test_Set_Used_For_Selection": False
        }])
    
        FINAL_CONFIG_PATH = (
            RESULTS_DIR /
            "P1_final_model_configuration.csv"
        )
    
        final_configuration.to_csv(
            FINAL_CONFIG_PATH,
            index=False
        )
    
        display(final_configuration)
    
        print("FINAL PROJECT 1 MODEL CONFIGURATION LOCKED.")
        print("Final threshold:", FINAL_THRESHOLD)
        print("Saved to:", FINAL_CONFIG_PATH)
else:
    print("Inspection mode: code not executed.")


**14. Final Model Evaluation on the held-out test set**

Following completion of model development and validation-based threshold selection, the locked E5 configuration was evaluated on the held-out 30% test subset, which had not been used for model or threshold selection. No further model, preprocessing or threshold modifications were made after examining the test results.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 115; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.1 Final Test Predictions
        # ============================================================
    
        FINAL_THRESHOLD = E5_OPTIMAL_THRESHOLD
    
        E5_test_probabilities = E5_model.predict(
            X_test_robust,
            batch_size=2048,
            verbose=0
        ).ravel()
    
        E5_test_predictions = (
            E5_test_probabilities >= FINAL_THRESHOLD
        ).astype(int)
    
        print("Final test predictions generated.")
        print(f"Locked decision threshold: {FINAL_THRESHOLD:.6f}")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 116; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.2 Final Test Metrics
        # ============================================================
    
        final_test_accuracy = accuracy_score(
            y_test,
            E5_test_predictions
        )
    
        final_test_precision = precision_score(
            y_test,
            E5_test_predictions,
            zero_division=0
        )
    
        final_test_recall = recall_score(
            y_test,
            E5_test_predictions,
            zero_division=0
        )
    
        final_test_f1 = f1_score(
            y_test,
            E5_test_predictions,
            zero_division=0
        )
    
        final_test_average_precision = average_precision_score(
            y_test,
            E5_test_probabilities
        )
    
        final_precision_curve, final_recall_curve, _ = (
            precision_recall_curve(
                y_test,
                E5_test_probabilities
            )
        )
    
        final_test_pr_auc = auc(
            final_recall_curve,
            final_precision_curve
        )
    
        final_test_roc_auc = roc_auc_score(
            y_test,
            E5_test_probabilities
        )
    
        final_tn, final_fp, final_fn, final_tp = confusion_matrix(
            y_test,
            E5_test_predictions
        ).ravel()
    
        final_test_metrics = pd.DataFrame({
            "Metric": [
                "Accuracy",
                "Precision",
                "Recall",
                "F1 Score",
                "PR-AUC",
                "Average Precision",
                "ROC-AUC",
                "True Negatives",
                "False Positives",
                "False Negatives",
                "True Positives"
            ],
            "Value": [
                final_test_accuracy,
                final_test_precision,
                final_test_recall,
                final_test_f1,
                final_test_pr_auc,
                final_test_average_precision,
                final_test_roc_auc,
                final_tn,
                final_fp,
                final_fn,
                final_tp
            ]
        })
    
        display(final_test_metrics)
    
        final_test_metrics.to_csv(
            RESULTS_DIR / "P1_final_test_metrics.csv",
            index=False
        )
    
        print("Final test metrics saved permanently.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 117; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.3 Save Final Test Predictions
        # ============================================================
    
        final_test_predictions_df = pd.DataFrame({
            "Actual_Class": y_test.to_numpy(),
            "Fraud_Probability": E5_test_probabilities,
            "Predicted_Class": E5_test_predictions
        })
    
        final_test_predictions_df.to_csv(
            RESULTS_DIR / "P1_final_test_predictions.csv",
            index=False
        )
    
        print("Final test predictions saved permanently.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 118; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.4 Final E5 Training / Validation Loss
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        epochs = np.arange(
            1,
            len(history_E5_df) + 1
        )
    
        ax.plot(
            epochs,
            history_E5_df["loss"],
            label="Training Loss"
        )
    
        ax.plot(
            epochs,
            history_E5_df["val_loss"],
            label="Validation Loss"
        )
    
        ax.set_title(
            "Final E5 DNN – Training and Validation Loss"
        )
    
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Binary Cross-Entropy Loss")
    
        ax.legend()
    
        plt.tight_layout()
    
        FINAL_LOSS_PATH = (
            FIGURE_DIR /
            "P1_final_E5_loss.png"
        )
    
        plt.savefig(
            FINAL_LOSS_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Final loss figure saved to:")
        print(FINAL_LOSS_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 119; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.5 Final E5 Training / Validation Accuracy
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.plot(
            epochs,
            history_E5_df["accuracy"],
            label="Training Accuracy"
        )
    
        ax.plot(
            epochs,
            history_E5_df["val_accuracy"],
            label="Validation Accuracy"
        )
    
        ax.set_title(
            "Final E5 DNN – Training and Validation Accuracy"
        )
    
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Accuracy")
    
        ax.legend()
    
        plt.tight_layout()
    
        FINAL_ACCURACY_PATH = (
            FIGURE_DIR /
            "P1_final_E5_accuracy.png"
        )
    
        plt.savefig(
            FINAL_ACCURACY_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Final accuracy figure saved to:")
        print(FINAL_ACCURACY_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 120; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.6 Final Test Precision-Recall Curve
        # ============================================================
    
        fig, ax = plt.subplots(figsize=(8, 5))
    
        ax.plot(
            final_recall_curve,
            final_precision_curve,
            label=f"E5 Test PR-AUC = {final_test_pr_auc:.4f}"
        )
    
        ax.set_title(
            "Final E5 DNN – Test Precision-Recall Curve"
        )
    
        ax.set_xlabel("Recall")
        ax.set_ylabel("Precision")
    
        ax.set_xlim(0, 1.05)
        ax.set_ylim(0, 1.05)
    
        ax.legend()
    
        plt.tight_layout()
    
        FINAL_PR_PATH = (
            FIGURE_DIR /
            "P1_final_E5_precision_recall_curve.png"
        )
    
        plt.savefig(
            FINAL_PR_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Final PR curve saved to:")
        print(FINAL_PR_PATH)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 121; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 14.7 Final Test Confusion Matrix
        # ============================================================
    
        from sklearn.metrics import ConfusionMatrixDisplay
    
        final_cm = confusion_matrix(
            y_test,
            E5_test_predictions
        )
    
        fig, ax = plt.subplots(figsize=(6, 5))
    
        cm_display = ConfusionMatrixDisplay(
            confusion_matrix=final_cm,
            display_labels=[
                "Legitimate",
                "Fraud"
            ]
        )
    
        cm_display.plot(
            ax=ax,
            values_format=",d",
            colorbar=False
        )
    
        ax.set_title(
            "Final E5 DNN – Test Confusion Matrix"
        )
    
        plt.tight_layout()
    
        FINAL_CM_PATH = (
            FIGURE_DIR /
            "P1_final_E5_confusion_matrix.png"
        )
    
        plt.savefig(
            FINAL_CM_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print("Final confusion matrix saved to:")
        print(FINAL_CM_PATH)
else:
    print("Inspection mode: code not executed.")


**15. Error Analysis**

Error analysis was conducted on the held-out test predictions after final model evaluation to examine the characteristics and confidence of incorrect classifications. Particular attention was given to false-negative cases because these represent fraudulent transactions that were not detected by the model. Since V1–V28 are anonymised PCA components, interpretation focuses on prediction confidence, transaction amount and time rather than assigning unsupported real-world meanings to individual transformed features.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 123; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.1 Build Final Test Error Analysis Dataset
        # ============================================================
    
        test_analysis = X_test.copy()
    
        # Preserve cleaned-dataset row index for traceability
        test_analysis.insert(
            0,
            "Cleaned_Row_Index",
            test_analysis.index
        )
    
        # Add actual class labels
        test_analysis["Actual_Class"] = (
            y_test.to_numpy()
        )
    
        # Add final E5 fraud probabilities
        test_analysis["Fraud_Probability"] = (
            E5_test_probabilities
        )
    
        # Add final predicted classes
        test_analysis["Predicted_Class"] = (
            E5_test_predictions
        )
    
        # Identify whether each prediction is correct
        test_analysis["Correct"] = (
            test_analysis["Actual_Class"]
            == test_analysis["Predicted_Class"]
        )
    
        # Assign confusion-matrix outcome labels
        conditions = [
            (
                (test_analysis["Actual_Class"] == 0)
                & (test_analysis["Predicted_Class"] == 0)
            ),
            (
                (test_analysis["Actual_Class"] == 0)
                & (test_analysis["Predicted_Class"] == 1)
            ),
            (
                (test_analysis["Actual_Class"] == 1)
                & (test_analysis["Predicted_Class"] == 0)
            ),
            (
                (test_analysis["Actual_Class"] == 1)
                & (test_analysis["Predicted_Class"] == 1)
            )
        ]
    
        outcomes = [
            "True Negative",
            "False Positive",
            "False Negative",
            "True Positive"
        ]
    
        test_analysis["Outcome"] = np.select(
            conditions,
            outcomes,
            default="Unknown"
        )
    
        print(
            "Final test error-analysis dataset created."
        )
    
        print(
            "\nOutcome distribution:"
        )
    
        print(
            test_analysis["Outcome"].value_counts()
        )
    
        display(
            test_analysis[
                [
                    "Cleaned_Row_Index",
                    "Time",
                    "Amount",
                    "Actual_Class",
                    "Fraud_Probability",
                    "Predicted_Class",
                    "Outcome"
                ]
            ].head()
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 124; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.2 Error Summary by Classification Outcome
        # ============================================================
    
        error_summary = (
            test_analysis
            .groupby("Outcome")
            .agg(
                Transactions=(
                    "Outcome",
                    "size"
                ),
                Mean_Amount=(
                    "Amount",
                    "mean"
                ),
                Median_Amount=(
                    "Amount",
                    "median"
                ),
                Mean_Fraud_Probability=(
                    "Fraud_Probability",
                    "mean"
                ),
                Median_Fraud_Probability=(
                    "Fraud_Probability",
                    "median"
                )
            )
            .round(4)
        )
    
        display(error_summary)
    
        error_summary.to_csv(
            RESULTS_DIR /
            "P1_final_error_summary.csv"
        )
    
        print("Final error summary saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 125; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.3 False Negative Analysis
        # ============================================================
    
        false_negatives = (
            test_analysis[
                test_analysis["Outcome"]
                == "False Negative"
            ]
            .copy()
        )
    
        false_negatives["Distance_From_Threshold"] = (
            FINAL_THRESHOLD
            - false_negatives["Fraud_Probability"]
        )
    
        false_negatives = (
            false_negatives
            .sort_values(
                "Fraud_Probability",
                ascending=False
            )
        )
    
        print(
            "Number of false negatives:",
            len(false_negatives)
        )
    
        print("\nClosest false negatives to the threshold:")
    
        display(
            false_negatives[
                [
                    "Cleaned_Row_Index",
                    "Time",
                    "Amount",
                    "Fraud_Probability",
                    "Distance_From_Threshold"
                ]
            ].head(10)
        )
    
        print("\nLowest-confidence fraud cases:")
    
        display(
            false_negatives[
                [
                    "Cleaned_Row_Index",
                    "Time",
                    "Amount",
                    "Fraud_Probability",
                    "Distance_From_Threshold"
                ]
            ].tail(10)
        )
    
        false_negatives.to_csv(
            RESULTS_DIR /
            "P1_final_false_negatives.csv",
            index=False
        )
    
        print("False-negative cases saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 126; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.4 False Positive Analysis
        # ============================================================
    
        false_positives = (
            test_analysis[
                test_analysis["Outcome"]
                == "False Positive"
            ]
            .copy()
        )
    
        false_positives["Distance_From_Threshold"] = (
            false_positives["Fraud_Probability"]
            - FINAL_THRESHOLD
        )
    
        false_positives = (
            false_positives
            .sort_values(
                "Fraud_Probability",
                ascending=False
            )
        )
    
        print(
            "Number of false positives:",
            len(false_positives)
        )
    
        display(
            false_positives[
                [
                    "Cleaned_Row_Index",
                    "Time",
                    "Amount",
                    "Fraud_Probability",
                    "Distance_From_Threshold"
                ]
            ]
        )
    
        false_positives_path = (
            RESULTS_DIR /
            "P1_final_false_positives.csv"
        )
    
        false_positives.to_csv(
            false_positives_path,
            index=False
        )
    
        print("False-positive cases saved.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 127; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.5 False Negative Confidence Summary
        # ============================================================
    
        fn_confidence_summary = pd.DataFrame({
            "Measure": [
                "False negatives",
                "Selected threshold",
                "Highest missed fraud probability",
                "Median missed fraud probability",
                "Lowest missed fraud probability",
                "Missed frauds with probability >= 0.40",
                "Missed frauds with probability < 0.10"
            ],
            "Value": [
                len(false_negatives),
                FINAL_THRESHOLD,
                false_negatives[
                    "Fraud_Probability"
                ].max(),
                false_negatives[
                    "Fraud_Probability"
                ].median(),
                false_negatives[
                    "Fraud_Probability"
                ].min(),
                int(
                    (
                        false_negatives[
                            "Fraud_Probability"
                        ] >= 0.40
                    ).sum()
                ),
                int(
                    (
                        false_negatives[
                            "Fraud_Probability"
                        ] < 0.10
                    ).sum()
                )
            ]
        })
    
        display(
            fn_confidence_summary
        )
    
        fn_confidence_summary.to_csv(
            RESULTS_DIR /
            "P1_false_negative_confidence_summary.csv",
            index=False
        )
    
        print(
            "False-negative confidence summary saved."
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 128; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.6 Transaction Amount: Detected vs Missed Fraud
        # ============================================================
    
        fraud_cases = test_analysis[
            test_analysis["Actual_Class"] == 1
        ].copy()
    
        fraud_amount_comparison = (
            fraud_cases
            .groupby("Outcome")["Amount"]
            .agg(
                [
                    "count",
                    "mean",
                    "median",
                    "min",
                    "max"
                ]
            )
            .round(2)
        )
    
        display(
            fraud_amount_comparison
        )
    
        fraud_amount_comparison.to_csv(
            RESULTS_DIR /
            "P1_fraud_amount_error_analysis.csv"
        )
    
        print(
            "Detected vs missed fraud amount analysis saved."
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 129; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 15.7 Save Complete Final Test Error Analysis
        # ============================================================
    
        ERROR_ANALYSIS_PATH = (
            RESULTS_DIR /
            "P1_complete_test_error_analysis.csv"
        )
    
        test_analysis.to_csv(
            ERROR_ANALYSIS_PATH,
            index=False
        )
    
        print(
            "Complete test error-analysis dataset "
            "saved permanently to:"
        )
    
        print(
            ERROR_ANALYSIS_PATH
        )
else:
    print("Inspection mode: code not executed.")


**16. Five Representative Inference Cases**

Five representative observations from the held-out test subset were selected after final model evaluation to demonstrate different behaviours of the final E5 fraud-detection model. The cases comprise a high-confidence true positive, a high-confidence true negative, a correct prediction close to the selected decision threshold, a false positive and a false negative. These observations were selected only after final model evaluation for explanatory purposes and did not influence model development or threshold selection.

The reported confidence values are derived from the model's sigmoid output for the predicted class and should be interpreted as model confidence scores rather than formally calibrated real-world probabilities.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 131; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.1 Select Five Representative Inference Cases
        # ============================================================
    
        # ------------------------------------------------------------
        # Case 1: Highest-confidence correctly detected fraud
        # ------------------------------------------------------------
    
        case_true_positive = (
            test_analysis[
                test_analysis["Outcome"] == "True Positive"
            ]
            .sort_values(
                "Fraud_Probability",
                ascending=False
            )
            .iloc[0]
        )
    
        # ------------------------------------------------------------
        # Case 2: Highest-confidence correctly classified legitimate
        # ------------------------------------------------------------
    
        case_true_negative = (
            test_analysis[
                test_analysis["Outcome"] == "True Negative"
            ]
            .sort_values(
                "Fraud_Probability",
                ascending=True
            )
            .iloc[0]
        )
    
        # ------------------------------------------------------------
        # Case 3: Correct prediction closest to selected threshold
        # ------------------------------------------------------------
    
        correct_cases = (
            test_analysis[
                test_analysis["Correct"] == True
            ]
            .copy()
        )
    
        correct_cases["Threshold_Distance"] = abs(
            correct_cases["Fraud_Probability"]
            - FINAL_THRESHOLD
        )
    
        case_borderline = (
            correct_cases
            .sort_values(
                "Threshold_Distance",
                ascending=True
            )
            .iloc[0]
        )
    
        # ------------------------------------------------------------
        # Case 4: Highest-confidence false positive
        # Legitimate transaction most strongly predicted as fraud
        # ------------------------------------------------------------
    
        case_false_positive = (
            test_analysis[
                test_analysis["Outcome"] == "False Positive"
            ]
            .sort_values(
                "Fraud_Probability",
                ascending=False
            )
            .iloc[0]
        )
    
        # ------------------------------------------------------------
        # Case 5: Highest-confidence false negative
        # Fraud transaction most strongly predicted as legitimate
        # ------------------------------------------------------------
    
        case_false_negative = (
            test_analysis[
                test_analysis["Outcome"] == "False Negative"
            ]
            .sort_values(
                "Fraud_Probability",
                ascending=True
            )
            .iloc[0]
        )
    
        # ------------------------------------------------------------
        # Combine the five selected cases
        # ------------------------------------------------------------
    
        selected_cases = pd.DataFrame([
            case_true_positive,
            case_true_negative,
            case_borderline,
            case_false_positive,
            case_false_negative
        ]).reset_index(drop=True)
    
        selected_cases.insert(
            0,
            "Case_ID",
            [
                "Case 1",
                "Case 2",
                "Case 3",
                "Case 4",
                "Case 5"
            ]
        )
    
        selected_cases.insert(
            1,
            "Case_Type",
            [
                "High-confidence True Positive",
                "High-confidence True Negative",
                "Borderline Correct Prediction",
                "High-confidence False Positive",
                "High-confidence False Negative"
            ]
        )
    
        print("Five representative inference cases selected successfully.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 132; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.2 Calculate Predicted-Class Confidence
        # ============================================================
    
        selected_cases["Prediction_Confidence"] = np.where(
            selected_cases["Predicted_Class"] == 1,
            selected_cases["Fraud_Probability"],
            1 - selected_cases["Fraud_Probability"]
        )
    
        selected_cases["Threshold_Distance"] = abs(
            selected_cases["Fraud_Probability"]
            - FINAL_THRESHOLD
        )
    
        selected_cases["Prediction_Confidence_Percent"] = (
            selected_cases["Prediction_Confidence"] * 100
        )
    
        print("Prediction confidence calculated.")
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 133; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.3 Five Inference Cases - Summary
        # ============================================================
    
        inference_case_summary = selected_cases[
            [
                "Case_ID",
                "Case_Type",
                "Cleaned_Row_Index",
                "Time",
                "Amount",
                "Fraud_Probability",
                "Predicted_Class",
                "Actual_Class",
                "Prediction_Confidence_Percent",
                "Threshold_Distance",
                "Outcome"
            ]
        ].copy()
    
        # Cleaner display formatting
        inference_case_summary[
            "Fraud_Probability"
        ] = inference_case_summary[
            "Fraud_Probability"
        ].round(6)
    
        inference_case_summary[
            "Prediction_Confidence_Percent"
        ] = inference_case_summary[
            "Prediction_Confidence_Percent"
        ].round(4)
    
        inference_case_summary[
            "Threshold_Distance"
        ] = inference_case_summary[
            "Threshold_Distance"
        ].round(6)
    
        display(inference_case_summary)
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 134; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.4 Verify Selected Inference Cases
        # ============================================================
    
        unique_case_count = (
            selected_cases["Cleaned_Row_Index"]
            .nunique()
        )
    
        print(
            "Unique selected observations:",
            unique_case_count
        )
    
        print(
            "Expected observations: 5"
        )
    
        if unique_case_count == 5:
            print("PASS: All five inference cases are unique.")
        else:
            print(
                "WARNING: Duplicate inference cases detected."
            )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 135; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.5 Save Five Representative Inference Cases
        # ============================================================
    
        INFERENCE_CASES_PATH = (
            RESULTS_DIR /
            "P1_five_inference_cases.csv"
        )
    
        selected_cases.to_csv(
            INFERENCE_CASES_PATH,
            index=False
        )
    
        inference_case_summary.to_csv(
            RESULTS_DIR /
            "P1_five_inference_cases_summary.csv",
            index=False
        )
    
        print(
            "Five inference cases saved permanently."
        )
    
        print(
            "Full cases:",
            INFERENCE_CASES_PATH
        )
    
        print(
            "Summary:",
            RESULTS_DIR /
            "P1_five_inference_cases_summary.csv"
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 136; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 16.6 Create Report-Friendly Five-Case Summary
        # ============================================================
    
        report_inference_cases = selected_cases[
            [
                "Case_ID",
                "Case_Type",
                "Cleaned_Row_Index",
                "Amount",
                "Fraud_Probability",
                "Predicted_Class",
                "Actual_Class",
                "Prediction_Confidence_Percent",
                "Outcome"
            ]
        ].copy()
    
        # Convert numeric classes to readable labels
        class_labels = {
            0: "Legitimate",
            1: "Fraud"
        }
    
        report_inference_cases["Predicted_Class"] = (
            report_inference_cases["Predicted_Class"]
            .map(class_labels)
        )
    
        report_inference_cases["Actual_Class"] = (
            report_inference_cases["Actual_Class"]
            .map(class_labels)
        )
    
        # Display probabilities with enough precision to avoid
        # very small probabilities appearing as exact zero
        report_inference_cases["Fraud_Probability"] = (
            report_inference_cases["Fraud_Probability"]
            .map(lambda x: f"{x:.8g}")
        )
    
        report_inference_cases[
            "Prediction_Confidence_Percent"
        ] = (
            report_inference_cases[
                "Prediction_Confidence_Percent"
            ].round(2)
        )
    
        display(report_inference_cases)
    
        report_inference_cases.to_csv(
            RESULTS_DIR /
            "P1_five_inference_cases_report_summary.csv",
            index=False
        )
    
        print("Report-friendly inference summary saved.")
else:
    print("Inspection mode: code not executed.")


**17. Explainability Analysis**

Explainability analysis was performed after the final E5 model and decision threshold had been locked. Global feature importance was estimated using permutation analysis, measuring the reduction in precision-recall performance when individual features were disrupted. Local feature sensitivity was subsequently examined for the five representative inference cases. Since V1–V28 are anonymised PCA components, the analysis identifies influential transformed features without assigning unsupported real-world interpretations to them.

In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 138; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.1 Explainability Baseline
        # ============================================================
    
        # Final E5 probabilities on validation data already exist as:
        # E5_val_probabilities
    
        explain_precision, explain_recall, _ = (
            precision_recall_curve(
                y_val,
                E5_val_probabilities
            )
        )
    
        EXPLAIN_BASELINE_PR_AUC = auc(
            explain_recall,
            explain_precision
        )
    
        feature_names = list(X_val.columns)
    
        print(
            "Explainability baseline validation PR-AUC:",
            round(EXPLAIN_BASELINE_PR_AUC, 6)
        )
    
        print(
            "Number of input features:",
            len(feature_names)
        )
    
        print(
            "Features:",
            feature_names
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 139; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.2 Global Permutation Importance
        # PR-AUC Decrease
        # ============================================================
    
        PERMUTATION_REPEATS = 3
    
        rng = np.random.default_rng(SEED)
    
        permutation_rows = []
    
        print(
            "Running permutation importance..."
        )
    
        for feature_index, feature_name in enumerate(feature_names):
    
            permuted_scores = []
    
            for repeat in range(PERMUTATION_REPEATS):
    
                X_permuted = (
                    X_val_robust.copy()
                )
    
                shuffled_values = (
                    X_permuted[:, feature_index]
                    .copy()
                )
    
                rng.shuffle(
                    shuffled_values
                )
    
                X_permuted[
                    :,
                    feature_index
                ] = shuffled_values
    
                permuted_probabilities = (
                    E5_model.predict(
                        X_permuted,
                        batch_size=2048,
                        verbose=0
                    )
                    .ravel()
                )
    
                perm_precision, perm_recall, _ = (
                    precision_recall_curve(
                        y_val,
                        permuted_probabilities
                    )
                )
    
                perm_pr_auc = auc(
                    perm_recall,
                    perm_precision
                )
    
                permuted_scores.append(
                    perm_pr_auc
                )
    
            mean_permuted_pr_auc = float(
                np.mean(
                    permuted_scores
                )
            )
    
            std_permuted_pr_auc = float(
                np.std(
                    permuted_scores
                )
            )
    
            importance = (
                EXPLAIN_BASELINE_PR_AUC
                - mean_permuted_pr_auc
            )
    
            permutation_rows.append({
                "Feature": feature_name,
                "Baseline_PR_AUC":
                    EXPLAIN_BASELINE_PR_AUC,
                "Mean_Permuted_PR_AUC":
                    mean_permuted_pr_auc,
                "Std_Permuted_PR_AUC":
                    std_permuted_pr_auc,
                "PR_AUC_Decrease":
                    importance
            })
    
            print(
                f"{feature_name}: "
                f"importance = {importance:.6f}"
            )
    
        permutation_importance_df = pd.DataFrame(
            permutation_rows
        )
    
        permutation_importance_df = (
            permutation_importance_df
            .sort_values(
                "PR_AUC_Decrease",
                ascending=False
            )
            .reset_index(drop=True)
        )
    
        display(
            permutation_importance_df
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 140; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.3 Save Global Permutation Importance
        # ============================================================
    
        PERMUTATION_IMPORTANCE_PATH = (
            RESULTS_DIR /
            "P1_E5_permutation_importance.csv"
        )
    
        permutation_importance_df.to_csv(
            PERMUTATION_IMPORTANCE_PATH,
            index=False
        )
    
        print(
            "Permutation importance saved permanently to:"
        )
    
        print(
            PERMUTATION_IMPORTANCE_PATH
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 141; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.4 Top 10 Global Features
        # ============================================================
    
        top_10_features = (
            permutation_importance_df
            .head(10)
            .copy()
        )
    
        display(
            top_10_features[
                [
                    "Feature",
                    "Mean_Permuted_PR_AUC",
                    "PR_AUC_Decrease"
                ]
            ]
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 142; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.5 Global Permutation Importance Figure
        # ============================================================
    
        plot_data = (
            top_10_features
            .sort_values(
                "PR_AUC_Decrease",
                ascending=True
            )
        )
    
        fig, ax = plt.subplots(
            figsize=(8, 6)
        )
    
        ax.barh(
            plot_data["Feature"],
            plot_data["PR_AUC_Decrease"]
        )
    
        ax.set_title(
            "Final E5 DNN – Global Permutation Feature Importance"
        )
    
        ax.set_xlabel(
            "Decrease in Validation PR-AUC After Permutation"
        )
    
        ax.set_ylabel(
            "Input Feature"
        )
    
        plt.tight_layout()
    
        GLOBAL_IMPORTANCE_PATH = (
            FIGURE_DIR /
            "P1_E5_global_permutation_importance.png"
        )
    
        plt.savefig(
            GLOBAL_IMPORTANCE_PATH,
            dpi=300,
            bbox_inches="tight"
        )
    
        plt.show()
    
        print(
            "Global feature-importance figure saved to:"
        )
    
        print(
            GLOBAL_IMPORTANCE_PATH
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 143; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.6 Local Feature-Occlusion Sensitivity
        # Five Representative Cases
        # ============================================================
    
        selected_indices = (
            selected_cases[
                "Cleaned_Row_Index"
            ]
            .astype(int)
            .tolist()
        )
    
        # Retrieve the original feature rows
        selected_raw_features = (
            X_test.loc[
                selected_indices
            ]
        )
    
        # Transform using the already-fitted RobustScaler
        selected_scaled_features = (
            robust_scaler.transform(
                selected_raw_features
            )
        )
    
        local_explanation_rows = []
    
        for case_position in range(
            len(selected_cases)
        ):
    
            case_id = selected_cases.loc[
                case_position,
                "Case_ID"
            ]
    
            case_type = selected_cases.loc[
                case_position,
                "Case_Type"
            ]
    
            original_probability = float(
                E5_model.predict(
                    selected_scaled_features[
                        case_position:
                        case_position + 1
                    ],
                    verbose=0
                )[0][0]
            )
    
            # Create 30 copies:
            # one copy for each feature being neutralised
            occluded_batch = np.repeat(
                selected_scaled_features[
                    case_position:
                    case_position + 1
                ],
                repeats=len(feature_names),
                axis=0
            )
    
            for feature_index in range(
                len(feature_names)
            ):
                # Zero represents approximately the training median
                # after RobustScaler centring
                occluded_batch[
                    feature_index,
                    feature_index
                ] = 0.0
    
            occluded_probabilities = (
                E5_model.predict(
                    occluded_batch,
                    batch_size=128,
                    verbose=0
                )
                .ravel()
            )
    
            for feature_index, feature_name in enumerate(
                feature_names
            ):
    
                probability_change = (
                    original_probability
                    - occluded_probabilities[
                        feature_index
                    ]
                )
    
                local_explanation_rows.append({
                    "Case_ID": case_id,
                    "Case_Type": case_type,
                    "Feature": feature_name,
                    "Original_Fraud_Probability":
                        original_probability,
                    "Probability_After_Occlusion":
                        float(
                            occluded_probabilities[
                                feature_index
                            ]
                        ),
                    "Probability_Change":
                        float(
                            probability_change
                        ),
                    "Absolute_Change":
                        abs(
                            float(
                                probability_change
                            )
                        )
                })
    
        local_explanations = pd.DataFrame(
            local_explanation_rows
        )
    
        print(
            "Local feature-occlusion analysis completed."
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 144; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.7 Top Five Local Features Per Inference Case
        # ============================================================
    
        top_local_features = (
            local_explanations
            .sort_values(
                [
                    "Case_ID",
                    "Absolute_Change"
                ],
                ascending=[
                    True,
                    False
                ]
            )
            .groupby(
                "Case_ID"
            )
            .head(5)
            .reset_index(drop=True)
        )
    
        display(
            top_local_features[
                [
                    "Case_ID",
                    "Case_Type",
                    "Feature",
                    "Original_Fraud_Probability",
                    "Probability_After_Occlusion",
                    "Probability_Change"
                ]
            ]
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 145; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.8 Save Local Explainability Results
        # ============================================================
    
        local_explanations.to_csv(
            RESULTS_DIR /
            "P1_E5_local_occlusion_explanations.csv",
            index=False
        )
    
        top_local_features.to_csv(
            RESULTS_DIR /
            "P1_E5_top_local_features.csv",
            index=False
        )
    
        print(
            "Local explainability results saved permanently."
        )
else:
    print("Inspection mode: code not executed.")


In [ ]:
if PORTFOLIO_EXECUTE:
    if SUBMISSION_MODE:
        print('Submission mode: skipped locked Project 1 cell 146; saved output below remains the submitted evidence.')
    else:
        # ============================================================
        # 17.9 Report-Friendly Local Explainability Summary
        # ============================================================
    
        report_local_explanations = (
            top_local_features[
                top_local_features["Case_ID"].isin(
                    ["Case 3", "Case 4", "Case 5"]
                )
            ][
                [
                    "Case_ID",
                    "Case_Type",
                    "Feature",
                    "Original_Fraud_Probability",
                    "Probability_After_Occlusion",
                    "Probability_Change"
                ]
            ]
            .copy()
        )
    
        report_local_explanations[
            "Original_Fraud_Probability"
        ] = report_local_explanations[
            "Original_Fraud_Probability"
        ].map(
            lambda x: f"{x:.8g}"
        )
    
        report_local_explanations[
            "Probability_After_Occlusion"
        ] = report_local_explanations[
            "Probability_After_Occlusion"
        ].map(
            lambda x: f"{x:.8g}"
        )
    
        report_local_explanations[
            "Probability_Change"
        ] = report_local_explanations[
            "Probability_Change"
        ].map(
            lambda x: f"{x:.8g}"
        )
    
        display(
            report_local_explanations
        )
    
        report_local_explanations.to_csv(
            RESULTS_DIR /
            "P1_E5_report_local_explainability.csv",
            index=False
        )
    
        print(
            "Report-friendly local explainability "
            "summary saved permanently."
        )
else:
    print("Inspection mode: code not executed.")
